In [ ]:
import pandas as pd
import numpy as np

In [ ]:
file_path = "/Users/isaacasante/Documents/offload/Research Projects/DATA/mHealth/Comment_divided/Doctor_comment_split[3].csv"

In [ ]:
import pandas as pd

file_path = "/Users/isaacasante/Documents/offload/Research Projects/DATA/mHealth/Comment_divided/Doctor_comment_split[3].csv"

df = pd.read_csv(
    file_path,
    encoding="utf-16",
    sep="\t",
    engine="python"
)

print("Dataset shape:", df.shape)
df.head()

In [ ]:
import pandas as pd
import numpy as np

required_columns = [
    "医生名字",
    "医生性别",
    "医生职称",
    "所属医院",
    "部门名称",
    "标注部门名称",
    "所在省份",
    "所在城市",
    "医生标识",
    "医生擅长",
    "医生简介",
    "评价数量",
    "患者数量",
    "评论标识",
    "评论时间",
    "评论内容",
    "评论昵称",
    "评论类型",
    "评论症状",
    "评论标签",
    "评论打分"
]

available_columns = df.columns.tolist()

audit_columns = pd.DataFrame({
    "column": required_columns,
    "available": [col in available_columns for col in required_columns],
    "missing_count": [
        df[col].isna().sum() if col in df.columns else None
        for col in required_columns
    ],
    "missing_percent": [
        round(df[col].isna().mean() * 100, 2) if col in df.columns else None
        for col in required_columns
    ],
    "unique_values": [
        df[col].nunique(dropna=True) if col in df.columns else None
        for col in required_columns
    ]
})

audit_columns

In [ ]:
# Patient-side need text
patient_cols = [col for col in ["评论症状", "评论标签", "评论内容"] if col in df.columns]

df["patient_need_text"] = (
    df[patient_cols]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Physician-side expertise text
physician_cols = [col for col in ["标注部门名称", "部门名称", "医生擅长", "医生简介"] if col in df.columns]

df["physician_expertise_text"] = (
    df[physician_cols]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df[["patient_need_text", "physician_expertise_text"]].head()

In [ ]:
df["patient_text_length"] = df["patient_need_text"].str.len()
df["physician_text_length"] = df["physician_expertise_text"].str.len()

text_usability = pd.DataFrame({
    "condition": [
        "patient_need_text length >= 5",
        "patient_need_text length >= 10",
        "patient_need_text length >= 20",
        "physician_expertise_text length >= 10",
        "physician_expertise_text length >= 20",
        "both patient >=10 and physician >=20"
    ],
    "count": [
        (df["patient_text_length"] >= 5).sum(),
        (df["patient_text_length"] >= 10).sum(),
        (df["patient_text_length"] >= 20).sum(),
        (df["physician_text_length"] >= 10).sum(),
        (df["physician_text_length"] >= 20).sum(),
        ((df["patient_text_length"] >= 10) & (df["physician_text_length"] >= 20)).sum()
    ]
})

text_usability["percent"] = round(text_usability["count"] / len(df) * 100, 2)

text_usability

In [ ]:
rating_audit = pd.DataFrame({
    "count": df["评论打分"].value_counts(dropna=False).sort_index(),
    "percent": round(df["评论打分"].value_counts(normalize=True, dropna=False).sort_index() * 100, 2)
})

rating_audit

In [ ]:
coverage = {}

for col in ["医生标识", "所属医院", "部门名称", "标注部门名称", "所在省份", "所在城市", "医生职称", "评论类型"]:
    if col in df.columns:
        coverage[col] = df[col].nunique(dropna=True)

coverage_df = pd.DataFrame(list(coverage.items()), columns=["variable", "unique_count"])

coverage_df

In [ ]:
dept_col = "标注部门名称" if "标注部门名称" in df.columns else "部门名称"

department_distribution = (
    df[dept_col]
    .value_counts(dropna=False)
    .reset_index()
)

department_distribution.columns = ["department", "review_count"]

department_distribution.head(30)

In [ ]:
analysis_df = df.copy()

conditions = []

conditions.append(analysis_df["评论打分"].notna())
conditions.append(analysis_df["医生标识"].notna())
conditions.append(analysis_df["patient_text_length"] >= 10)
conditions.append(analysis_df["physician_text_length"] >= 20)

final_condition = np.logical_and.reduce(conditions)

analysis_df = analysis_df[final_condition].copy()

print("Original dataset size:", len(df))
print("Analysis-ready dataset size:", len(analysis_df))
print("Retained percent:", round(len(analysis_df) / len(df) * 100, 2))

analysis_df.head()

In [ ]:
import re
import pandas as pd

# Function to remove characters that Excel cannot handle
def clean_excel_illegal_chars(value):
    if isinstance(value, str):
        # Remove illegal XML control characters
        return re.sub(r"[\x00-\x08\x0B\x0C\x0E-\x1F]", "", value)
    return value

# Clean all audit tables before writing to Excel
audit_columns_clean = audit_columns.map(clean_excel_illegal_chars)
text_usability_clean = text_usability.applymap(clean_excel_illegal_chars)
rating_audit_clean = rating_audit.applymap(clean_excel_illegal_chars)
coverage_df_clean = coverage_df.applymap(clean_excel_illegal_chars)
department_distribution_clean = department_distribution.applymap(clean_excel_illegal_chars)

# Save Excel audit report
with pd.ExcelWriter("paper4_data_audit.xlsx", engine="openpyxl") as writer:
    audit_columns_clean.to_excel(writer, sheet_name="column_missingness", index=False)
    text_usability_clean.to_excel(writer, sheet_name="text_usability", index=False)
    rating_audit_clean.to_excel(writer, sheet_name="rating_distribution")
    coverage_df_clean.to_excel(writer, sheet_name="coverage", index=False)
    department_distribution_clean.to_excel(writer, sheet_name="department_distribution", index=False)

# Save analysis-ready dataset as CSV
analysis_df.to_csv(
    "paper4_analysis_ready_dataset.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved: paper4_data_audit.xlsx")
print("Saved: paper4_analysis_ready_dataset.csv")

In [ ]:
coverage_df

In [ ]:
department_distribution.head(30)

In [ ]:
import numpy as np
import pandas as pd

# Ensure rating is numeric
analysis_df["rating"] = pd.to_numeric(analysis_df["评论打分"], errors="coerce")

# Create binary outcome
analysis_df["lower_rating"] = np.where(analysis_df["rating"] < 5, 1, 0)

# Split into lower-rating and five-star groups
lower_df = analysis_df[analysis_df["lower_rating"] == 1].copy()
high_df = analysis_df[analysis_df["lower_rating"] == 0].copy()

print("Available lower-rating cases:", len(lower_df))
print("Available five-star cases:", len(high_df))

# Desired sample sizes
n_lower = 10000
n_high = 40000

# If fewer lower-rating cases are available, keep all and adjust high-rating sample
n_lower_actual = min(n_lower, len(lower_df))
n_high_actual = 50000 - n_lower_actual

lower_sample = lower_df.sample(
    n=n_lower_actual,
    random_state=42
)

high_sample = high_df.sample(
    n=n_high_actual,
    random_state=42
)

paper4_sample_50000 = pd.concat([lower_sample, high_sample], axis=0)

# Shuffle the final sample
paper4_sample_50000 = paper4_sample_50000.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Final sample size:", len(paper4_sample_50000))
print(paper4_sample_50000["lower_rating"].value_counts())
print(round(paper4_sample_50000["lower_rating"].value_counts(normalize=True) * 100, 2))

# Save sample
paper4_sample_50000.to_csv(
    "paper4_sample_50000_stratified.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved: paper4_sample_50000_stratified.csv")

In [ ]:
paper4_sample_50000_random = analysis_df.sample(
    n=50000,
    random_state=42
).reset_index(drop=True)

paper4_sample_50000_random.to_csv(
    "paper4_sample_50000_random.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Random sample rating distribution:")
print(paper4_sample_50000_random["lower_rating"].value_counts())
print(round(paper4_sample_50000_random["lower_rating"].value_counts(normalize=True) * 100, 2))

In [ ]:
import pandas as pd

file_path = "paper4_sample_50000_stratified.csv"

model_df = pd.read_csv(file_path, encoding="utf-8-sig")

print("Model dataset shape:", model_df.shape)
model_df.head()

In [ ]:
# Patient-side text for alignment: exclude 评论标签 to avoid satisfaction-tag leakage
model_df["patient_need_text_main"] = (
    model_df[["评论症状", "评论内容"]]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Physician-side expertise text
physician_cols = [
    col for col in ["标注部门名称", "部门名称", "医生擅长", "医生简介"]
    if col in model_df.columns
]

model_df["physician_expertise_text_main"] = (
    model_df[physician_cols]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

model_df[["patient_need_text_main", "physician_expertise_text_main"]].head()

In [ ]:
import os

print(os.getcwd())
print(os.listdir())

In [ ]:
from pathlib import Path

project_dir = Path("/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit")

folders = [
    project_dir,
    project_dir / "data",
    project_dir / "data" / "raw",
    project_dir / "data" / "processed",
    project_dir / "data" / "samples",
    project_dir / "outputs",
    project_dir / "outputs" / "tables",
    project_dir / "outputs" / "figures",
    project_dir / "scripts",
    project_dir / "notebooks"
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

print("Created project folder:")
print(project_dir)

In [ ]:
import os

os.chdir(project_dir)

print("Current working directory:")
print(os.getcwd())

In [ ]:
import shutil
from pathlib import Path

old_dir = Path("/Users/isaacasante/Documents/offload/Research Projects/DATA/mHealth/Comment_divided")
project_dir = Path("/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit")

files_to_copy = [
    "paper4_data_audit.xlsx",
    "paper4_analysis_ready_dataset.csv",
    "paper4_sample_50000_stratified.csv",
    "paper4_sample_50000_random.csv"
]

for file in files_to_copy:
    src = old_dir / file
    dst = project_dir / "data" / "processed" / file
    
    if src.exists():
        shutil.copy2(src, dst)
        print(f"Copied: {file}")
    else:
        print(f"Not found: {src}")

In [ ]:
import os

print("Current working directory:")
print(os.getcwd())

print("\nFiles here:")
for f in os.listdir():
    if "paper4" in f.lower():
        print(f)

In [ ]:
from pathlib import Path

search_root = Path("/Users/isaacasante/Documents/offload/Research Projects")

paper4_files = list(search_root.rglob("paper4_*"))

print("Found files:")
for f in paper4_files:
    print(f)

In [ ]:
import shutil
from pathlib import Path

project_dir = Path("/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit")

destination_dir = project_dir / "data" / "processed"
destination_dir.mkdir(parents=True, exist_ok=True)

# Use the files found by the search
for src in paper4_files:
    dst = destination_dir / src.name
    shutil.copy2(src, dst)
    print(f"Copied: {src.name}")

In [ ]:
from pathlib import Path

project_dir = Path("/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit")
processed_dir = project_dir / "data" / "processed"

print("Processed folder exists:", processed_dir.exists())
print("Processed folder path:", processed_dir)

print("\nFiles in processed folder:")
for f in processed_dir.iterdir():
    print(f.name, "—", round(f.stat().st_size / (1024 * 1024), 2), "MB")

In [ ]:
import os

print("Current working directory:")
print(os.getcwd())

print("\nPaper4 files in current folder:")
for f in os.listdir():
    if "paper4" in f.lower():
        print(f)

In [ ]:
from pathlib import Path

search_root = Path("/Users/isaacasante")

matches = list(search_root.rglob("paper4_sample_50000_stratified.csv"))

print("Found matches:")
for m in matches:
    print(m)

In [ ]:
from pathlib import Path

project_dir = Path("/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit")
processed_dir = project_dir / "data" / "processed"

print("Processed folder exists:", processed_dir.exists())
print("Processed folder path:", processed_dir)

print("\nFiles in processed folder:")
for f in processed_dir.iterdir():
    print(f.name, "—", round(f.stat().st_size / (1024 * 1024), 2), "MB")

In [ ]:
expected_files = [
    "paper4_data_audit.xlsx",
    "paper4_analysis_ready_dataset.csv",
    "paper4_sample_50000_stratified.csv",
    "paper4_sample_50000_random.csv"
]

for file in expected_files:
    path = processed_dir / file
    if path.exists():
        print(f"FOUND: {file} — {round(path.stat().st_size / (1024 * 1024), 2)} MB")
    else:
        print(f"MISSING: {file}")

In [ ]:
from pathlib import Path

project_dir = Path("/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit")

print("Files in main project folder:")
for f in project_dir.iterdir():
    print(f.name)

In [ ]:
from pathlib import Path
import shutil

project_dir = Path("/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit")

# Create folders if they do not exist
(project_dir / "data" / "processed").mkdir(parents=True, exist_ok=True)
(project_dir / "data" / "samples").mkdir(parents=True, exist_ok=True)
(project_dir / "outputs" / "tables").mkdir(parents=True, exist_ok=True)

move_plan = {
    "paper4_analysis_ready_dataset.csv": project_dir / "data" / "processed",
    "paper4_sample_50000_stratified.csv": project_dir / "data" / "samples",
    "paper4_sample_50000_random.csv": project_dir / "data" / "samples",
    "paper4_data_audit.xlsx": project_dir / "outputs" / "tables",
}

for filename, destination_folder in move_plan.items():
    src = project_dir / filename
    dst = destination_folder / filename

    if src.exists():
        shutil.move(str(src), str(dst))
        print(f"Moved: {filename} → {destination_folder}")
    elif dst.exists():
        print(f"Already in correct folder: {filename}")
    else:
        print(f"Not found: {filename}")

In [ ]:
from pathlib import Path

project_dir = Path("/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit")

expected_paths = [
    project_dir / "outputs" / "tables" / "paper4_data_audit.xlsx",
    project_dir / "data" / "processed" / "paper4_analysis_ready_dataset.csv",
    project_dir / "data" / "samples" / "paper4_sample_50000_stratified.csv",
    project_dir / "data" / "samples" / "paper4_sample_50000_random.csv",
]

for path in expected_paths:
    if path.exists():
        print(f"FOUND: {path.name} — {round(path.stat().st_size / (1024 * 1024), 2)} MB")
    else:
        print(f"MISSING: {path}")

In [ ]:
import pandas as pd
from pathlib import Path

project_dir = Path("/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit")

sample_path = project_dir / "data" / "samples" / "paper4_sample_50000_stratified.csv"

model_df = pd.read_csv(sample_path, encoding="utf-8-sig")

print("Loaded successfully.")
print("Shape:", model_df.shape)
print(model_df["lower_rating"].value_counts())
print(round(model_df["lower_rating"].value_counts(normalize=True) * 100, 2))

model_df.head()

In [ ]:
# Patient-side text for alignment: exclude 评论标签 to avoid satisfaction/evaluation leakage
model_df["patient_need_text_main"] = (
    model_df[["评论症状", "评论内容"]]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Physician-side expertise text
physician_cols = [
    col for col in ["标注部门名称", "部门名称", "医生擅长", "医生简介"]
    if col in model_df.columns
]

model_df["physician_expertise_text_main"] = (
    model_df[physician_cols]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Check the revised text fields
model_df[[
    "评论症状",
    "评论内容",
    "评论标签",
    "patient_need_text_main",
    "physician_expertise_text_main"
]].head(10)

In [ ]:
model_df["patient_need_length_main"] = model_df["patient_need_text_main"].str.len()
model_df["physician_expertise_length_main"] = model_df["physician_expertise_text_main"].str.len()

text_check = pd.DataFrame({
    "condition": [
        "patient_need_text_main length >= 5",
        "patient_need_text_main length >= 10",
        "physician_expertise_text_main length >= 20",
        "both patient >=10 and physician >=20"
    ],
    "count": [
        (model_df["patient_need_length_main"] >= 5).sum(),
        (model_df["patient_need_length_main"] >= 10).sum(),
        (model_df["physician_expertise_length_main"] >= 20).sum(),
        ((model_df["patient_need_length_main"] >= 10) & 
         (model_df["physician_expertise_length_main"] >= 20)).sum()
    ]
})

text_check["percent"] = round(text_check["count"] / len(model_df) * 100, 2)

text_check

In [ ]:
from pathlib import Path

project_dir = Path("/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit")

save_path = project_dir / "data" / "samples" / "paper4_sample_50000_stratified_cleantext.csv"

model_df.to_csv(save_path, index=False, encoding="utf-8-sig")

print("Saved:", save_path)

In [ ]:
model_df_final = model_df[
    (model_df["patient_need_length_main"] >= 10) &
    (model_df["physician_expertise_length_main"] >= 20)
].copy()

print("Final modeling sample:", model_df_final.shape)
print(model_df_final["lower_rating"].value_counts())
print(round(model_df_final["lower_rating"].value_counts(normalize=True) * 100, 2))

In [ ]:
from pathlib import Path

project_dir = Path("/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit")

save_path = project_dir / "data" / "samples" / "paper4_final_model_sample_46674.csv"

model_df_final.to_csv(save_path, index=False, encoding="utf-8-sig")

print("Saved:", save_path)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import pandas as pd
import numpy as np

tfidf_df = model_df_final.copy()

patient_text = tfidf_df["patient_need_text_main"].fillna("").astype(str)
physician_text = tfidf_df["physician_expertise_text_main"].fillna("").astype(str)

combined_text = pd.concat([patient_text, physician_text], axis=0)

vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=5
)

vectorizer.fit(combined_text)

patient_tfidf = vectorizer.transform(patient_text)
physician_tfidf = vectorizer.transform(physician_text)

alignment_scores = []

batch_size = 5000

for start in tqdm(range(0, tfidf_df.shape[0], batch_size)):
    end = min(start + batch_size, tfidf_df.shape[0])
    sim = cosine_similarity(
        patient_tfidf[start:end],
        physician_tfidf[start:end]
    ).diagonal()
    alignment_scores.extend(sim)

tfidf_df["alignment_tfidf"] = alignment_scores

tfidf_df["alignment_tfidf"].describe()

In [ ]:
from pathlib import Path

project_dir = Path("/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit")

save_path = project_dir / "data" / "processed" / "paper4_model_dataset_with_tfidf_alignment.csv"

tfidf_df.to_csv(save_path, index=False, encoding="utf-8-sig")

print("Saved:", save_path)

In [ ]:
tfidf_df["alignment_tfidf"].describe()

In [ ]:
zero_count = (tfidf_df["alignment_tfidf"] == 0).sum()
zero_percent = zero_count / len(tfidf_df) * 100

print("Zero TF-IDF alignment count:", zero_count)
print("Zero TF-IDF alignment percent:", round(zero_percent, 2))

print("\nNon-zero TF-IDF alignment summary:")
print(tfidf_df.loc[tfidf_df["alignment_tfidf"] > 0, "alignment_tfidf"].describe())

In [ ]:
!pip install sentence-transformers

In [ ]:
import pandas as pd
from pathlib import Path

project_dir = Path("/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit")

tfidf_df = pd.read_csv(
    project_dir / "data" / "processed" / "paper4_model_dataset_with_tfidf_alignment.csv",
    encoding="utf-8-sig"
)

print(tfidf_df.shape)
tfidf_df.head()

In [ ]:
!pip install sentence-transformers

In [ ]:
import sys

print(sys.executable)
print(sys.version)

In [ ]:
import sys

print(sys.executable)
print(sys.version)

In [ ]:
import sys

print(sys.executable)
print(sys.version)

In [ ]:
import sys

!{sys.executable} -m pip install torch torchvision torchaudio

In [ ]:
import sys

print(sys.executable)
print(sys.version)

In [ ]:
import sys

!{sys.executable} -m pip install sentence-transformers transformers torch accelerate

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("Model loaded successfully.")

In [ ]:
import sys

!{sys.executable} -m pip uninstall -y sentence-transformers transformers accelerate

In [ ]:
import sys

!{sys.executable} -m pip install "sentence-transformers==2.7.0" "transformers==4.41.2" "accelerate==0.30.1"

In [ ]:
import sys
print(sys.executable)
print(sys.version)

In [ ]:
import torch
print(torch.__version__)
print(torch.backends.mps.is_available())

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("Model loaded successfully.")

In [ ]:
import pandas as pd
from pathlib import Path

project_dir = Path("/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit")

tfidf_df = pd.read_csv(
    project_dir / "data" / "processed" / "paper4_model_dataset_with_tfidf_alignment.csv",
    encoding="utf-8-sig"
)

print(tfidf_df.shape)
tfidf_df.head()

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import torch

device = "mps" if torch.backends.mps.is_available() else "cpu"
print("Using device:", device)

model = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2",
    device=device
)

patient_texts = tfidf_df["patient_need_text_main"].fillna("").astype(str).tolist()
physician_texts = tfidf_df["physician_expertise_text_main"].fillna("").astype(str).tolist()

patient_embeddings = model.encode(
    patient_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

physician_embeddings = model.encode(
    physician_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

tfidf_df["alignment_llm"] = np.sum(patient_embeddings * physician_embeddings, axis=1)

tfidf_df["alignment_llm"].describe()

In [ ]:
save_path = project_dir / "data" / "processed" / "paper4_model_dataset_with_tfidf_llm_alignment.csv"

tfidf_df.to_csv(save_path, index=False, encoding="utf-8-sig")

print("Saved:", save_path)

In [ ]:
tfidf_df["alignment_llm"].describe()

In [ ]:
alignment_by_rating = tfidf_df.groupby("lower_rating")["alignment_llm"].describe()

alignment_by_rating

In [ ]:
tfidf_df.groupby("lower_rating")["alignment_llm"].agg(
    count="count",
    mean="mean",
    median="median",
    std="std",
    min="min",
    max="max"
)

In [ ]:
tfidf_df["alignment_llm_z"] = (
    tfidf_df["alignment_llm"] - tfidf_df["alignment_llm"].mean()
) / tfidf_df["alignment_llm"].std()

tfidf_df["alignment_tfidf_z"] = (
    tfidf_df["alignment_tfidf"] - tfidf_df["alignment_tfidf"].mean()
) / tfidf_df["alignment_tfidf"].std()

In [ ]:
from pathlib import Path

project_dir = Path("/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit")

save_path = project_dir / "data" / "processed" / "paper4_model_dataset_with_tfidf_llm_alignment.csv"

tfidf_df.to_csv(save_path, index=False, encoding="utf-8-sig")

print("Saved:", save_path)

In [ ]:
from scipy.stats import ttest_ind, mannwhitneyu

high_alignment = tfidf_df.loc[tfidf_df["lower_rating"] == 0, "alignment_llm"]
low_alignment = tfidf_df.loc[tfidf_df["lower_rating"] == 1, "alignment_llm"]

# t-test
t_stat, t_p = ttest_ind(high_alignment, low_alignment, equal_var=False)

# Mann-Whitney U test
u_stat, u_p = mannwhitneyu(high_alignment, low_alignment, alternative="two-sided")

print("Welch t-test")
print("t-statistic:", t_stat)
print("p-value:", t_p)

print("\nMann-Whitney U test")
print("U-statistic:", u_stat)
print("p-value:", u_p)

In [ ]:
from pathlib import Path

project_dir = Path("/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit")

save_path = project_dir / "data" / "processed" / "paper4_model_dataset_with_tfidf_llm_alignment.csv"

tfidf_df.to_csv(save_path, index=False, encoding="utf-8-sig")

print("Saved:", save_path)

In [ ]:
import numpy as np
import pandas as pd

# Standardize alignment variables
tfidf_df["alignment_llm_z"] = (
    tfidf_df["alignment_llm"] - tfidf_df["alignment_llm"].mean()
) / tfidf_df["alignment_llm"].std()

tfidf_df["alignment_tfidf_z"] = (
    tfidf_df["alignment_tfidf"] - tfidf_df["alignment_tfidf"].mean()
) / tfidf_df["alignment_tfidf"].std()

# Log-transform platform reputation variables
tfidf_df["log_review_count"] = np.log1p(pd.to_numeric(tfidf_df["评价数量"], errors="coerce"))
tfidf_df["log_patient_count"] = np.log1p(pd.to_numeric(tfidf_df["患者数量"], errors="coerce"))

# Date variables
tfidf_df["评论时间"] = pd.to_datetime(tfidf_df["评论时间"], errors="coerce")
tfidf_df["review_year"] = tfidf_df["评论时间"].dt.year

# Basic text length control
tfidf_df["patient_need_length_main"] = tfidf_df["patient_need_text_main"].fillna("").astype(str).str.len()

# Check variables
tfidf_df[[
    "lower_rating",
    "alignment_llm_z",
    "alignment_tfidf_z",
    "log_review_count",
    "log_patient_count",
    "医生职称",
    "标注部门名称",
    "所在省份",
    "评论类型",
    "review_year",
    "patient_need_length_main"
]].head()

In [ ]:
import statsmodels.formula.api as smf

model1 = smf.logit(
    formula="lower_rating ~ alignment_llm_z",
    data=tfidf_df
).fit()

print(model1.summary())

In [ ]:
import sys

!{sys.executable} -m pip install statsmodels

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

project_dir = Path("/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit")

tfidf_df = pd.read_csv(
    project_dir / "data" / "processed" / "paper4_model_dataset_with_tfidf_llm_alignment.csv",
    encoding="utf-8-sig"
)

print(tfidf_df.shape)
tfidf_df.head()

In [ ]:
tfidf_df["alignment_llm_z"] = (
    tfidf_df["alignment_llm"] - tfidf_df["alignment_llm"].mean()
) / tfidf_df["alignment_llm"].std()

tfidf_df["alignment_tfidf_z"] = (
    tfidf_df["alignment_tfidf"] - tfidf_df["alignment_tfidf"].mean()
) / tfidf_df["alignment_tfidf"].std()

tfidf_df["log_review_count"] = np.log1p(pd.to_numeric(tfidf_df["评价数量"], errors="coerce"))
tfidf_df["log_patient_count"] = np.log1p(pd.to_numeric(tfidf_df["患者数量"], errors="coerce"))

tfidf_df["评论时间"] = pd.to_datetime(tfidf_df["评论时间"], errors="coerce")
tfidf_df["review_year"] = tfidf_df["评论时间"].dt.year

tfidf_df["patient_need_length_main"] = (
    tfidf_df["patient_need_text_main"]
    .fillna("")
    .astype(str)
    .str.len()
)

In [ ]:
import statsmodels.formula.api as smf

model1 = smf.logit(
    formula="lower_rating ~ alignment_llm_z",
    data=tfidf_df
).fit()

print(model1.summary())

In [ ]:
import statsmodels.formula.api as smf

model1 = smf.logit(
    formula="lower_rating ~ alignment_llm_z",
    data=tfidf_df
).fit(maxiter=200)

print(model1.summary())

In [ ]:
model2a = smf.logit(
    formula="""
    lower_rating ~ alignment_llm_z
    + log_review_count
    + log_patient_count
    + patient_need_length_main
    """,
    data=tfidf_df
).fit(maxiter=200)

print(model2a.summary())

In [ ]:
model2b = smf.logit(
    formula="""
    lower_rating ~ alignment_llm_z
    + log_review_count
    + log_patient_count
    + patient_need_length_main
    + C(评论类型)
    + C(所在省份)
    """,
    data=tfidf_df
).fit(maxiter=200)

print(model2b.summary())

In [ ]:
# Create cleaned modeling copy
reg_df = tfidf_df.copy()

# Keep departments with at least 300 observations
dept_counts = reg_df["标注部门名称"].value_counts()
major_depts = dept_counts[dept_counts >= 300].index

reg_df["dept_group"] = reg_df["标注部门名称"].where(
    reg_df["标注部门名称"].isin(major_depts),
    "Other"
)

# Group rare physician titles
title_counts = reg_df["医生职称"].value_counts()
major_titles = title_counts[title_counts >= 300].index

reg_df["title_group"] = reg_df["医生职称"].where(
    reg_df["医生职称"].isin(major_titles),
    "Other"
)

# Check
print("Department groups:", reg_df["dept_group"].nunique())
print("Title groups:", reg_df["title_group"].nunique())
print(reg_df["dept_group"].value_counts().head())
print(reg_df["title_group"].value_counts())

In [ ]:
model3 = smf.logit(
    formula="""
    lower_rating ~ alignment_llm_z
    + log_review_count
    + log_patient_count
    + patient_need_length_main
    + C(title_group)
    + C(dept_group)
    + C(所在省份)
    + C(评论类型)
    """,
    data=reg_df
).fit(maxiter=200)

print(model3.summary())

In [ ]:
reg_df = tfidf_df.copy()

# Group departments with fewer than 300 observations
dept_counts = reg_df["标注部门名称"].value_counts()
major_depts = dept_counts[dept_counts >= 300].index

reg_df["dept_group"] = reg_df["标注部门名称"].where(
    reg_df["标注部门名称"].isin(major_depts),
    "Other"
)

# Group physician titles with fewer than 300 observations
title_counts = reg_df["医生职称"].value_counts()
major_titles = title_counts[title_counts >= 300].index

reg_df["title_group"] = reg_df["医生职称"].where(
    reg_df["医生职称"].isin(major_titles),
    "Other"
)

# Group provinces with fewer than 500 observations
province_counts = reg_df["所在省份"].value_counts()
major_provinces = province_counts[province_counts >= 500].index

reg_df["province_group"] = reg_df["所在省份"].where(
    reg_df["所在省份"].isin(major_provinces),
    "Other"
)

print("Department groups:", reg_df["dept_group"].nunique())
print("Title groups:", reg_df["title_group"].nunique())
print("Province groups:", reg_df["province_group"].nunique())

print(reg_df["province_group"].value_counts())

In [ ]:
reg_df = tfidf_df.copy()

# Group departments with fewer than 300 observations
dept_counts = reg_df["标注部门名称"].value_counts()
major_depts = dept_counts[dept_counts >= 300].index

reg_df["dept_group"] = reg_df["标注部门名称"].where(
    reg_df["标注部门名称"].isin(major_depts),
    "Other"
)

# Group physician titles with fewer than 300 observations
title_counts = reg_df["医生职称"].value_counts()
major_titles = title_counts[title_counts >= 300].index

reg_df["title_group"] = reg_df["医生职称"].where(
    reg_df["医生职称"].isin(major_titles),
    "Other"
)

# Group provinces with fewer than 500 observations
province_counts = reg_df["所在省份"].value_counts()
major_provinces = province_counts[province_counts >= 500].index

reg_df["province_group"] = reg_df["所在省份"].where(
    reg_df["所在省份"].isin(major_provinces),
    "Other"
)

print("Department groups:", reg_df["dept_group"].nunique())
print("Title groups:", reg_df["title_group"].nunique())
print("Province groups:", reg_df["province_group"].nunique())

print(reg_df["province_group"].value_counts())

In [ ]:
province_outcome_check = pd.crosstab(
    reg_df["province_group"],
    reg_df["lower_rating"],
    margins=True
)

province_outcome_check

In [ ]:
import statsmodels.formula.api as smf

model4 = smf.logit(
    formula="""
    lower_rating ~ alignment_llm_z
    + log_review_count
    + log_patient_count
    + patient_need_length_main
    + C(title_group)
    + C(dept_group)
    + C(province_group)
    + C(评论类型)
    """,
    data=reg_df
).fit(method="lbfgs", maxiter=500)

print(model4.summary())

In [ ]:
import numpy as np

coef = model4.params["alignment_llm_z"]
odds_ratio = np.exp(coef)

print("alignment_llm_z coefficient:", coef)
print("odds ratio:", odds_ratio)
print("percent change in odds:", round((odds_ratio - 1) * 100, 2), "%")
print("p-value:", model4.pvalues["alignment_llm_z"])

In [ ]:
import numpy as np
import pandas as pd

def odds_ratio_table(model):
    params = model.params
    conf = model.conf_int()
    
    return pd.DataFrame({
        "coef": params,
        "odds_ratio": np.exp(params),
        "ci_lower": np.exp(conf[0]),
        "ci_upper": np.exp(conf[1]),
        "p_value": model.pvalues
    })

or_model1 = odds_ratio_table(model1)
or_model2 = odds_ratio_table(model2a)
or_model3 = odds_ratio_table(model2b)

or_model1.loc[["alignment_llm_z"]]

In [ ]:
or_model2.loc[["alignment_llm_z"]]

In [ ]:
or_model3.loc[["alignment_llm_z"]]

In [ ]:
or_model4 = odds_ratio_table(model4)
or_model4.loc[["alignment_llm_z"]]

In [ ]:
model4_reg = smf.logit(
    formula="""
    lower_rating ~ alignment_llm_z
    + log_review_count
    + log_patient_count
    + patient_need_length_main
    + C(title_group)
    + C(dept_group)
    + C(province_group)
    + C(评论类型)
    """,
    data=reg_df
).fit_regularized(alpha=0.01, maxiter=500)

print(model4_reg.summary())

coef = model4_reg.params["alignment_llm_z"]
odds_ratio = np.exp(coef)

print("alignment_llm_z coefficient:", coef)
print("odds ratio:", odds_ratio)
print("percent change in odds:", round((odds_ratio - 1) * 100, 2), "%")

In [ ]:
reg_df2 = tfidf_df.copy()

# Group departments with fewer than 1000 observations
dept_counts = reg_df2["标注部门名称"].value_counts()
major_depts = dept_counts[dept_counts >= 1000].index

reg_df2["dept_group_simple"] = reg_df2["标注部门名称"].where(
    reg_df2["标注部门名称"].isin(major_depts),
    "Other"
)

# Group physician titles with fewer than 1000 observations
title_counts = reg_df2["医生职称"].value_counts()
major_titles = title_counts[title_counts >= 1000].index

reg_df2["title_group_simple"] = reg_df2["医生职称"].where(
    reg_df2["医生职称"].isin(major_titles),
    "Other"
)

# Use the province grouping you already created, or recreate it
province_counts = reg_df2["所在省份"].value_counts()
major_provinces = province_counts[province_counts >= 500].index

reg_df2["province_group"] = reg_df2["所在省份"].where(
    reg_df2["所在省份"].isin(major_provinces),
    "Other"
)

print("Department groups:", reg_df2["dept_group_simple"].nunique())
print("Title groups:", reg_df2["title_group_simple"].nunique())
print("Province groups:", reg_df2["province_group"].nunique())

In [ ]:
model4_simple = smf.logit(
    formula="""
    lower_rating ~ alignment_llm_z
    + log_review_count
    + log_patient_count
    + patient_need_length_main
    + C(title_group_simple)
    + C(dept_group_simple)
    + C(province_group)
    + C(评论类型)
    """,
    data=reg_df2
).fit(method="lbfgs", maxiter=500)

print(model4_simple.summary())

coef = model4_simple.params["alignment_llm_z"]
odds_ratio = np.exp(coef)

print("alignment_llm_z coefficient:", coef)
print("odds ratio:", odds_ratio)
print("percent change in odds:", round((odds_ratio - 1) * 100, 2), "%")
print("p-value:", model4_simple.pvalues["alignment_llm_z"])

In [ ]:
from pathlib import Path

project_dir = Path("/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit")

save_path = project_dir / "data" / "processed" / "paper4_final_regression_dataset.csv"

reg_df2.to_csv(save_path, index=False, encoding="utf-8-sig")

print("Saved:", save_path)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

project_dir = Path("/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit")

df = pd.read_csv(
    project_dir / "data" / "processed" / "paper4_model_dataset_with_tfidf_llm_alignment.csv",
    encoding="utf-8-sig"
)

print(df.shape)
df.head()

In [ ]:
df["alignment_llm_z"] = (df["alignment_llm"] - df["alignment_llm"].mean()) / df["alignment_llm"].std()
df["alignment_tfidf_z"] = (df["alignment_tfidf"] - df["alignment_tfidf"].mean()) / df["alignment_tfidf"].std()

df["log_review_count"] = np.log1p(pd.to_numeric(df["评价数量"], errors="coerce"))
df["log_patient_count"] = np.log1p(pd.to_numeric(df["患者数量"], errors="coerce"))

df["patient_need_length_main"] = (
    df["patient_need_text_main"].fillna("").astype(str).str.len()
)

df["评论时间"] = pd.to_datetime(df["评论时间"], errors="coerce")
df["review_year"] = df["评论时间"].dt.year

In [ ]:
corr_vars = [
    "lower_rating",
    "alignment_llm",
    "alignment_tfidf",
    "log_review_count",
    "log_patient_count",
    "patient_need_length_main"
]

corr_matrix = df[corr_vars].corr()

corr_matrix

In [ ]:
corr_matrix.to_excel(
    project_dir / "outputs" / "tables" / "paper4_correlation_matrix.xlsx"
)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

vif_vars = [
    "alignment_llm_z",
    "alignment_tfidf_z",
    "log_review_count",
    "log_patient_count",
    "patient_need_length_main"
]

vif_data = df[vif_vars].dropna()
X = add_constant(vif_data)

vif_table = pd.DataFrame({
    "variable": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
})

vif_table

In [ ]:
vif_table.to_excel(
    project_dir / "outputs" / "tables" / "paper4_vif_table.xlsx",
    index=False
)

In [ ]:
import statsmodels.formula.api as smf

model_tfidf_llm = smf.logit(
    formula="""
    lower_rating ~ alignment_llm_z + alignment_tfidf_z
    + log_review_count
    + log_patient_count
    + patient_need_length_main
    + C(评论类型)
    + C(所在省份)
    """,
    data=df
).fit(maxiter=200)

print(model_tfidf_llm.summary())

In [ ]:
import statsmodels.formula.api as smf
import numpy as np
import pandas as pd

compare_df = df.copy()

# Create province grouping
province_counts = compare_df["所在省份"].value_counts()
major_provinces = province_counts[province_counts >= 500].index

compare_df["province_group"] = compare_df["所在省份"].where(
    compare_df["所在省份"].isin(major_provinces),
    "Other"
)

# Make sure standardized variables exist
compare_df["alignment_llm_z"] = (
    compare_df["alignment_llm"] - compare_df["alignment_llm"].mean()
) / compare_df["alignment_llm"].std()

compare_df["alignment_tfidf_z"] = (
    compare_df["alignment_tfidf"] - compare_df["alignment_tfidf"].mean()
) / compare_df["alignment_tfidf"].std()

model_tfidf_llm = smf.logit(
    formula="""
    lower_rating ~ alignment_llm_z + alignment_tfidf_z
    + log_review_count
    + log_patient_count
    + patient_need_length_main
    + C(评论类型)
    + C(province_group)
    """,
    data=compare_df
).fit(method="lbfgs", maxiter=500)

print(model_tfidf_llm.summary())

In [ ]:
def odds_ratio_table(model):
    params = model.params
    conf = model.conf_int()
    return pd.DataFrame({
        "coef": params,
        "odds_ratio": np.exp(params),
        "ci_lower": np.exp(conf[0]),
        "ci_upper": np.exp(conf[1]),
        "p_value": model.pvalues
    })

or_tfidf_llm = odds_ratio_table(model_tfidf_llm)

or_tfidf_llm.loc[["alignment_llm_z", "alignment_tfidf_z"]]

In [ ]:
model_tfidf_llm_simple = smf.logit(
    formula="""
    lower_rating ~ alignment_llm_z + alignment_tfidf_z
    + log_review_count
    + log_patient_count
    + patient_need_length_main
    """,
    data=compare_df
).fit(maxiter=200)

print(model_tfidf_llm_simple.summary())

or_tfidf_llm_simple = odds_ratio_table(model_tfidf_llm_simple)
or_tfidf_llm_simple.loc[["alignment_llm_z", "alignment_tfidf_z"]]

In [ ]:
import statsmodels.formula.api as smf
import numpy as np
import pandas as pd

compare_df = df.copy()

# Create province grouping
province_counts = compare_df["所在省份"].value_counts()
major_provinces = province_counts[province_counts >= 500].index

compare_df["province_group"] = compare_df["所在省份"].where(
    compare_df["所在省份"].isin(major_provinces),
    "Other"
)

# Make sure standardized variables exist
compare_df["alignment_llm_z"] = (
    compare_df["alignment_llm"] - compare_df["alignment_llm"].mean()
) / compare_df["alignment_llm"].std()

compare_df["alignment_tfidf_z"] = (
    compare_df["alignment_tfidf"] - compare_df["alignment_tfidf"].mean()
) / compare_df["alignment_tfidf"].std()

model_tfidf_llm = smf.logit(
    formula="""
    lower_rating ~ alignment_llm_z + alignment_tfidf_z
    + log_review_count
    + log_patient_count
    + patient_need_length_main
    + C(评论类型)
    + C(province_group)
    """,
    data=compare_df
).fit(method="lbfgs", maxiter=500)

print(model_tfidf_llm.summary())

In [ ]:
def odds_ratio_table(model):
    params = model.params
    conf = model.conf_int()
    return pd.DataFrame({
        "coef": params,
        "odds_ratio": np.exp(params),
        "ci_lower": np.exp(conf[0]),
        "ci_upper": np.exp(conf[1]),
        "p_value": model.pvalues
    })

or_tfidf_llm = odds_ratio_table(model_tfidf_llm)

or_tfidf_llm.loc[["alignment_llm_z", "alignment_tfidf_z"]]

In [ ]:
model_tfidf_llm_simple = smf.logit(
    formula="""
    lower_rating ~ alignment_llm_z + alignment_tfidf_z
    + log_review_count
    + log_patient_count
    + patient_need_length_main
    """,
    data=compare_df
).fit(maxiter=200)

print(model_tfidf_llm_simple.summary())

or_tfidf_llm_simple = odds_ratio_table(model_tfidf_llm_simple)
or_tfidf_llm_simple.loc[["alignment_llm_z", "alignment_tfidf_z"]]

In [ ]:
marginal_effects = model_tfidf_llm.get_margeff(
    at="overall",
    method="dydx"
)

print(marginal_effects.summary())

In [ ]:
import matplotlib.pyplot as plt

plot_df = df.copy()

alignment_range = np.linspace(
    plot_df["alignment_llm_z"].quantile(0.05),
    plot_df["alignment_llm_z"].quantile(0.95),
    50
)

base = plot_df.median(numeric_only=True)

pred_df = pd.DataFrame({
    "alignment_llm_z": alignment_range,
    "alignment_tfidf_z": 0,
    "log_review_count": base["log_review_count"],
    "log_patient_count": base["log_patient_count"],
    "patient_need_length_main": base["patient_need_length_main"],
    "评论类型": plot_df["评论类型"].mode()[0],
    "所在省份": plot_df["所在省份"].mode()[0]
})

pred_df["predicted_lower_rating"] = model_tfidf_llm.predict(pred_df)

plt.figure(figsize=(7, 5))
plt.plot(pred_df["alignment_llm_z"], pred_df["predicted_lower_rating"])
plt.xlabel("LLM-derived symptom–specialty alignment (standardized)")
plt.ylabel("Predicted probability of below-5-star rating")
plt.title("Predicted lower-rating probability by patient–physician alignment")
plt.tight_layout()

plt.savefig(
    project_dir / "outputs" / "figures" / "paper4_predicted_probability_alignment.png",
    dpi=300
)

plt.show()

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

project_dir = Path("/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit")

df = pd.read_csv(
    project_dir / "data" / "processed" / "paper4_model_dataset_with_tfidf_llm_alignment.csv",
    encoding="utf-8-sig"
)

print(df.shape)
df.head()

In [ ]:
df["alignment_llm_z"] = (df["alignment_llm"] - df["alignment_llm"].mean()) / df["alignment_llm"].std()
df["alignment_tfidf_z"] = (df["alignment_tfidf"] - df["alignment_tfidf"].mean()) / df["alignment_tfidf"].std()

df["log_review_count"] = np.log1p(pd.to_numeric(df["评价数量"], errors="coerce"))
df["log_patient_count"] = np.log1p(pd.to_numeric(df["患者数量"], errors="coerce"))

df["patient_need_length_main"] = (
    df["patient_need_text_main"].fillna("").astype(str).str.len()
)

df["评论时间"] = pd.to_datetime(df["评论时间"], errors="coerce")
df["review_year"] = df["评论时间"].dt.year

In [ ]:
corr_vars = [
    "lower_rating",
    "alignment_llm",
    "alignment_tfidf",
    "log_review_count",
    "log_patient_count",
    "patient_need_length_main"
]

corr_matrix = df[corr_vars].corr()

corr_matrix

In [ ]:
corr_matrix.to_excel(
    project_dir / "outputs" / "tables" / "paper4_correlation_matrix.xlsx"
)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

vif_vars = [
    "alignment_llm_z",
    "alignment_tfidf_z",
    "log_review_count",
    "log_patient_count",
    "patient_need_length_main"
]

vif_data = df[vif_vars].dropna()
X = add_constant(vif_data)

vif_table = pd.DataFrame({
    "variable": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
})

vif_table

In [ ]:
vif_table.to_excel(
    project_dir / "outputs" / "tables" / "paper4_vif_table.xlsx",
    index=False
)

In [ ]:
import statsmodels.formula.api as smf
import numpy as np
import pandas as pd

compare_df = df.copy()

# Create province grouping
province_counts = compare_df["所在省份"].value_counts()
major_provinces = province_counts[province_counts >= 500].index

compare_df["province_group"] = compare_df["所在省份"].where(
    compare_df["所在省份"].isin(major_provinces),
    "Other"
)

# Make sure standardized variables exist
compare_df["alignment_llm_z"] = (
    compare_df["alignment_llm"] - compare_df["alignment_llm"].mean()
) / compare_df["alignment_llm"].std()

compare_df["alignment_tfidf_z"] = (
    compare_df["alignment_tfidf"] - compare_df["alignment_tfidf"].mean()
) / compare_df["alignment_tfidf"].std()

model_tfidf_llm = smf.logit(
    formula="""
    lower_rating ~ alignment_llm_z + alignment_tfidf_z
    + log_review_count
    + log_patient_count
    + patient_need_length_main
    + C(评论类型)
    + C(province_group)
    """,
    data=compare_df
).fit(method="lbfgs", maxiter=500)

print(model_tfidf_llm.summary())

In [ ]:
def odds_ratio_table(model):
    params = model.params
    conf = model.conf_int()
    return pd.DataFrame({
        "coef": params,
        "odds_ratio": np.exp(params),
        "ci_lower": np.exp(conf[0]),
        "ci_upper": np.exp(conf[1]),
        "p_value": model.pvalues
    })

or_tfidf_llm = odds_ratio_table(model_tfidf_llm)

or_tfidf_llm.loc[["alignment_llm_z", "alignment_tfidf_z"]]

In [ ]:
model_tfidf_llm_simple = smf.logit(
    formula="""
    lower_rating ~ alignment_llm_z + alignment_tfidf_z
    + log_review_count
    + log_patient_count
    + patient_need_length_main
    """,
    data=compare_df
).fit(maxiter=200)

print(model_tfidf_llm_simple.summary())

or_tfidf_llm_simple = odds_ratio_table(model_tfidf_llm_simple)
or_tfidf_llm_simple.loc[["alignment_llm_z", "alignment_tfidf_z"]]

In [ ]:
marginal_effects = model_tfidf_llm.get_margeff(
    at="overall",
    method="dydx"
)

print(marginal_effects.summary())

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import sys
!{sys.executable} -m pip install matplotlib

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Project directory
project_dir = Path(
    "/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit"
)

# Reload dataset
df = pd.read_csv(
    project_dir / "data" / "processed" / "paper4_model_dataset_with_tfidf_llm_alignment.csv",
    encoding="utf-8-sig"
)

# Recreate key variables
df["alignment_llm_z"] = (
    df["alignment_llm"] - df["alignment_llm"].mean()
) / df["alignment_llm"].std()

df["alignment_tfidf_z"] = (
    df["alignment_tfidf"] - df["alignment_tfidf"].mean()
) / df["alignment_tfidf"].std()

df["log_review_count"] = np.log1p(
    pd.to_numeric(df["评价数量"], errors="coerce")
)

df["log_patient_count"] = np.log1p(
    pd.to_numeric(df["患者数量"], errors="coerce")
)

df["patient_need_length_main"] = (
    df["patient_need_text_main"]
    .fillna("")
    .astype(str)
    .str.len()
)

# Province grouping
province_counts = df["所在省份"].value_counts()
major_provinces = province_counts[province_counts >= 500].index

df["province_group"] = df["所在省份"].where(
    df["所在省份"].isin(major_provinces),
    "Other"
)

print("Dataset loaded.")
print(df.shape)

df.head()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plot_df = df.copy()

alignment_range = np.linspace(
    plot_df["alignment_llm_z"].quantile(0.05),
    plot_df["alignment_llm_z"].quantile(0.95),
    50
)

base = plot_df.median(numeric_only=True)

# Use the modal categories actually used in the fitted model
modal_review_type = plot_df["评论类型"].mode()[0]
modal_province_group = plot_df["province_group"].mode()[0]

pred_df = pd.DataFrame({
    "alignment_llm_z": alignment_range,
    "alignment_tfidf_z": 0,
    "log_review_count": base["log_review_count"],
    "log_patient_count": base["log_patient_count"],
    "patient_need_length_main": base["patient_need_length_main"],
    "评论类型": modal_review_type,
    "province_group": modal_province_group
})

pred_df["predicted_lower_rating"] = model_tfidf_llm.predict(pred_df)

plt.figure(figsize=(7, 5))
plt.plot(pred_df["alignment_llm_z"], pred_df["predicted_lower_rating"])
plt.xlabel("LLM-derived symptom–specialty alignment (standardized)")
plt.ylabel("Predicted probability of below-5-star rating")
plt.title("Predicted lower-rating probability by patient–physician alignment")
plt.tight_layout()

plt.savefig(
    project_dir / "outputs" / "figures" / "paper4_predicted_probability_alignment.png",
    dpi=300
)

plt.show()

In [ ]:
import statsmodels.formula.api as smf

model_tfidf_llm = smf.logit(
    formula="""
    lower_rating ~ alignment_llm_z + alignment_tfidf_z
    + log_review_count
    + log_patient_count
    + patient_need_length_main
    + C(评论类型)
    + C(province_group)
    """,
    data=df
).fit(method="lbfgs", maxiter=500)

print("Model refitted.")

In [ ]:
from pathlib import Path

project_dir = Path("/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit")

tables_dir = project_dir / "outputs" / "tables"
figures_dir = project_dir / "outputs" / "figures"

tables_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

print("Tables:", tables_dir)
print("Figures:", figures_dir)

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    project_dir / "data" / "processed" / "paper4_model_dataset_with_tfidf_llm_alignment.csv",
    encoding="utf-8-sig"
)

df["alignment_llm_z"] = (df["alignment_llm"] - df["alignment_llm"].mean()) / df["alignment_llm"].std()
df["alignment_tfidf_z"] = (df["alignment_tfidf"] - df["alignment_tfidf"].mean()) / df["alignment_tfidf"].std()

df["log_review_count"] = np.log1p(pd.to_numeric(df["评价数量"], errors="coerce"))
df["log_patient_count"] = np.log1p(pd.to_numeric(df["患者数量"], errors="coerce"))
df["patient_need_length_main"] = df["patient_need_text_main"].fillna("").astype(str).str.len()

province_counts = df["所在省份"].value_counts()
major_provinces = province_counts[province_counts >= 500].index
df["province_group"] = df["所在省份"].where(df["所在省份"].isin(major_provinces), "Other")

print(df.shape)

In [ ]:
desc_vars = [
    "lower_rating",
    "alignment_llm",
    "alignment_tfidf",
    "log_review_count",
    "log_patient_count",
    "patient_need_length_main"
]

table1 = df[desc_vars].describe().T
table1 = table1[["count", "mean", "std", "min", "25%", "50%", "75%", "max"]]

table1.to_excel(tables_dir / "Table1_descriptive_statistics.xlsx")
table1

In [ ]:
corr_vars = [
    "lower_rating",
    "alignment_llm",
    "alignment_tfidf",
    "log_review_count",
    "log_patient_count",
    "patient_need_length_main"
]

table2 = df[corr_vars].corr()

table2.to_excel(tables_dir / "Table2_correlation_matrix.xlsx")
table2

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

vif_vars = [
    "alignment_llm_z",
    "alignment_tfidf_z",
    "log_review_count",
    "log_patient_count",
    "patient_need_length_main"
]

X = add_constant(df[vif_vars].dropna())

table3 = pd.DataFrame({
    "variable": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
})

table3.to_excel(tables_dir / "Table3_vif.xlsx", index=False)
table3

In [ ]:
import statsmodels.formula.api as smf

model1 = smf.logit(
    "lower_rating ~ alignment_llm_z",
    data=df
).fit(maxiter=200)

model2 = smf.logit(
    """
    lower_rating ~ alignment_llm_z
    + log_review_count
    + log_patient_count
    + patient_need_length_main
    """,
    data=df
).fit(maxiter=200)

model3 = smf.logit(
    """
    lower_rating ~ alignment_llm_z
    + log_review_count
    + log_patient_count
    + patient_need_length_main
    + C(评论类型)
    + C(province_group)
    """,
    data=df
).fit(method="lbfgs", maxiter=500)

model4 = smf.logit(
    """
    lower_rating ~ alignment_llm_z
    + alignment_tfidf_z
    + log_review_count
    + log_patient_count
    + patient_need_length_main
    + C(评论类型)
    + C(province_group)
    """,
    data=df
).fit(method="lbfgs", maxiter=500)

In [ ]:
def extract_or(model, model_name):
    params = model.params
    conf = model.conf_int()
    pvals = model.pvalues
    
    out = pd.DataFrame({
        "variable": params.index,
        f"{model_name}_coef": params.values,
        f"{model_name}_OR": np.exp(params.values),
        f"{model_name}_CI_low": np.exp(conf[0].values),
        f"{model_name}_CI_high": np.exp(conf[1].values),
        f"{model_name}_p": pvals.values
    })
    return out

or1 = extract_or(model1, "M1")
or2 = extract_or(model2, "M2")
or3 = extract_or(model3, "M3")
or4 = extract_or(model4, "M4")

table4 = or1.merge(or2, on="variable", how="outer")
table4 = table4.merge(or3, on="variable", how="outer")
table4 = table4.merge(or4, on="variable", how="outer")

model_stats = pd.DataFrame({
    "statistic": ["N", "Log likelihood", "Pseudo R2"],
    "M1": [model1.nobs, model1.llf, model1.prsquared],
    "M2": [model2.nobs, model2.llf, model2.prsquared],
    "M3": [model3.nobs, model3.llf, model3.prsquared],
    "M4": [model4.nobs, model4.llf, model4.prsquared]
})

table4.to_excel(tables_dir / "Table4_logistic_regression_odds_ratios.xlsx", index=False)
model_stats.to_excel(tables_dir / "Table4_model_statistics.xlsx", index=False)

table4.head()

In [ ]:
main_effects = pd.DataFrame({
    "Model": ["M1", "M2", "M3", "M4"],
    "Coefficient": [
        model1.params["alignment_llm_z"],
        model2.params["alignment_llm_z"],
        model3.params["alignment_llm_z"],
        model4.params["alignment_llm_z"]
    ],
    "Odds ratio": [
        np.exp(model1.params["alignment_llm_z"]),
        np.exp(model2.params["alignment_llm_z"]),
        np.exp(model3.params["alignment_llm_z"]),
        np.exp(model4.params["alignment_llm_z"])
    ],
    "Percent change in odds": [
        (np.exp(model1.params["alignment_llm_z"]) - 1) * 100,
        (np.exp(model2.params["alignment_llm_z"]) - 1) * 100,
        (np.exp(model3.params["alignment_llm_z"]) - 1) * 100,
        (np.exp(model4.params["alignment_llm_z"]) - 1) * 100
    ],
    "p-value": [
        model1.pvalues["alignment_llm_z"],
        model2.pvalues["alignment_llm_z"],
        model3.pvalues["alignment_llm_z"],
        model4.pvalues["alignment_llm_z"]
    ]
})

main_effects.to_excel(tables_dir / "Table5_main_alignment_effects.xlsx", index=False)
main_effects

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 5))
plt.hist(df["alignment_llm"], bins=50)
plt.xlabel("LLM-derived symptom–specialty alignment")
plt.ylabel("Frequency")
plt.title("Distribution of LLM-derived alignment scores")
plt.tight_layout()
plt.savefig(figures_dir / "Figure1_llm_alignment_distribution.png", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
plt.hist(df["alignment_tfidf"], bins=50)
plt.xlabel("TF-IDF symptom–specialty alignment")
plt.ylabel("Frequency")
plt.title("Distribution of TF-IDF alignment scores")
plt.tight_layout()
plt.savefig(figures_dir / "Figure2_tfidf_alignment_distribution.png", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
df.boxplot(column="alignment_llm", by="lower_rating")
plt.suptitle("")
plt.title("LLM-derived alignment by rating group")
plt.xlabel("Lower rating group: 0 = five-star, 1 = below five-star")
plt.ylabel("LLM-derived alignment")
plt.tight_layout()
plt.savefig(figures_dir / "Figure3_alignment_by_rating_group.png", dpi=300)
plt.show()

In [ ]:
alignment_range = np.linspace(
    df["alignment_llm_z"].quantile(0.05),
    df["alignment_llm_z"].quantile(0.95),
    50
)

base = df.median(numeric_only=True)

pred_df = pd.DataFrame({
    "alignment_llm_z": alignment_range,
    "alignment_tfidf_z": 0,
    "log_review_count": base["log_review_count"],
    "log_patient_count": base["log_patient_count"],
    "patient_need_length_main": base["patient_need_length_main"],
    "评论类型": df["评论类型"].mode()[0],
    "province_group": df["province_group"].mode()[0]
})

pred_df["predicted_lower_rating"] = model4.predict(pred_df)

plt.figure(figsize=(7, 5))
plt.plot(pred_df["alignment_llm_z"], pred_df["predicted_lower_rating"])
plt.xlabel("LLM-derived symptom–specialty alignment (standardized)")
plt.ylabel("Predicted probability of below-5-star rating")
plt.title("Predicted lower-rating probability by patient–physician alignment")
plt.tight_layout()
plt.savefig(figures_dir / "Figure4_predicted_probability_alignment.png", dpi=300)
plt.show()

In [ ]:
key_vars = [
    "Intercept",
    "alignment_llm_z",
    "alignment_tfidf_z",
    "log_review_count",
    "log_patient_count",
    "patient_need_length_main"
]

clean_table4 = table4[table4["variable"].isin(key_vars)].copy()

clean_table4.to_excel(
    tables_dir / "Table4_clean_logistic_regression_key_predictors.xlsx",
    index=False
)

clean_table4

In [ ]:
model_specs = pd.DataFrame({
    "Model": ["M1", "M2", "M3", "M4"],
    "LLM alignment": ["Yes", "Yes", "Yes", "Yes"],
    "TF-IDF alignment": ["No", "No", "No", "Yes"],
    "Platform controls": ["No", "Yes", "Yes", "Yes"],
    "Review type controls": ["No", "No", "Yes", "Yes"],
    "Province controls": ["No", "No", "Yes", "Yes"],
    "N": [model1.nobs, model2.nobs, model3.nobs, model4.nobs],
    "Pseudo R2": [model1.prsquared, model2.prsquared, model3.prsquared, model4.prsquared],
    "Log likelihood": [model1.llf, model2.llf, model3.llf, model4.llf]
})

model_specs.to_excel(
    tables_dir / "Table4_model_specifications.xlsx",
    index=False
)

model_specs

In [ ]:
import pandas as pd
import numpy as np

# Clean names for key variables
var_labels = {
    "alignment_llm_z": "LLM alignment",
    "alignment_tfidf_z": "TF-IDF alignment",
    "log_review_count": "Log review count",
    "log_patient_count": "Log patient count",
    "patient_need_length_main": "Patient text length"
}

key_vars = list(var_labels.keys())

def stars(p):
    if pd.isna(p):
        return ""
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return ""

def format_or_ci(or_value, ci_low, ci_high, p_value):
    if pd.isna(or_value):
        return ""
    return f"{or_value:.3f}{stars(p_value)} [{ci_low:.3f}, {ci_high:.3f}]"

# Build clean table from your existing table4
clean_rows = []

for var in key_vars:
    row = {"Variable": var_labels[var]}
    
    for model in ["M1", "M2", "M3", "M4"]:
        or_col = f"{model}_OR"
        low_col = f"{model}_CI_low"
        high_col = f"{model}_CI_high"
        p_col = f"{model}_p"
        
        match = table4[table4["variable"] == var]
        
        if len(match) == 0:
            row[model] = ""
        else:
            match = match.iloc[0]
            row[model] = format_or_ci(
                match.get(or_col),
                match.get(low_col),
                match.get(high_col),
                match.get(p_col)
            )
    
    clean_rows.append(row)

clean_table4 = pd.DataFrame(clean_rows)

# Add model statistics rows
clean_table4 = pd.concat([
    clean_table4,
    pd.DataFrame([
        {
            "Variable": "Platform controls",
            "M1": "No", "M2": "Yes", "M3": "Yes", "M4": "Yes"
        },
        {
            "Variable": "Review type controls",
            "M1": "No", "M2": "No", "M3": "Yes", "M4": "Yes"
        },
        {
            "Variable": "Province controls",
            "M1": "No", "M2": "No", "M3": "Yes", "M4": "Yes"
        },
        {
            "Variable": "N",
            "M1": int(model1.nobs),
            "M2": int(model2.nobs),
            "M3": int(model3.nobs),
            "M4": int(model4.nobs)
        },
        {
            "Variable": "Pseudo R²",
            "M1": f"{model1.prsquared:.3f}",
            "M2": f"{model2.prsquared:.3f}",
            "M3": f"{model3.prsquared:.3f}",
            "M4": f"{model4.prsquared:.3f}"
        }
    ])
], ignore_index=True)

# Save
clean_table4.to_excel(
    tables_dir / "Table4_clean_manuscript_regression.xlsx",
    index=False
)

clean_table4

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from pathlib import Path

project_dir = Path("/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit")
figures_dir = project_dir / "outputs" / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(12, 7))
ax.set_xlim(0, 12)
ax.set_ylim(0, 7)
ax.axis("off")

def add_box(x, y, w, h, text, fontsize=11):
    box = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.03,rounding_size=0.08",
        linewidth=1.5,
        facecolor="white",
        edgecolor="black"
    )
    ax.add_patch(box)
    ax.text(
        x + w / 2, y + h / 2, text,
        ha="center", va="center",
        fontsize=fontsize,
        wrap=True
    )

def add_arrow(x1, y1, x2, y2):
    arrow = FancyArrowPatch(
        (x1, y1), (x2, y2),
        arrowstyle="->",
        mutation_scale=18,
        linewidth=1.5,
        color="black"
    )
    ax.add_patch(arrow)

# Input boxes
add_box(
    0.7, 4.7, 2.8, 1.1,
    "Patient-side text\nSymptoms + review content"
)

add_box(
    0.7, 2.8, 2.8, 1.1,
    "Physician-side text\nDepartment + expertise profile"
)

# Method box
add_box(
    4.4, 3.7, 3.2, 1.3,
    "LLM embedding model\nSemantic representation of\npatient need and physician expertise"
)

# Alignment construct
add_box(
    8.4, 3.7, 2.9, 1.3,
    "Symptom–specialty\nalignment score\nCosine similarity"
)

# Outcome
add_box(
    8.4, 1.4, 2.9, 1.2,
    "Patient satisfaction outcome\nBelow-5-star rating"
)

# Controls
add_box(
    4.4, 0.7, 3.2, 1.2,
    "Control variables\nReview count, patient count,\nreview type, province, text length"
)

# Arrows
add_arrow(3.5, 5.25, 4.4, 4.55)
add_arrow(3.5, 3.35, 4.4, 4.15)
add_arrow(7.6, 4.35, 8.4, 4.35)
add_arrow(9.85, 3.7, 9.85, 2.6)
add_arrow(7.6, 1.3, 8.4, 1.9)

# Title
ax.text(
    6, 6.45,
    "Conceptual framework: LLM-derived patient–physician fit in online healthcare",
    ha="center",
    va="center",
    fontsize=15,
    fontweight="bold"
)

# Footnote
ax.text(
    6, 0.2,
    "Note. The main hypothesis is that higher symptom–specialty alignment is associated with lower odds of below-5-star ratings.",
    ha="center",
    va="center",
    fontsize=10
)

plt.tight_layout()

save_path = figures_dir / "Figure1_conceptual_framework.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", save_path)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from pathlib import Path

project_dir = Path("/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit")
figures_dir = project_dir / "outputs" / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(12, 7))
ax.set_xlim(0, 12)
ax.set_ylim(0, 7)
ax.axis("off")

def add_box(x, y, w, h, text, fontsize=11):
    box = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.04,rounding_size=0.08",
        linewidth=1.5,
        facecolor="white",
        edgecolor="black"
    )
    ax.add_patch(box)
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center",
            fontsize=fontsize, wrap=True)

def add_arrow(x1, y1, x2, y2):
    ax.add_patch(FancyArrowPatch(
        (x1, y1), (x2, y2),
        arrowstyle="->",
        mutation_scale=18,
        linewidth=1.5,
        color="black"
    ))

# Boxes
add_box(0.7, 4.7, 2.8, 1.1, "Patient-side text\nSymptoms + review content")
add_box(0.7, 2.8, 2.8, 1.1, "Physician-side text\nDepartment + expertise profile")

add_box(
    4.2, 3.7, 3.5, 1.3,
    "LLM embedding model\nSemantic representation of\npatient needs and\nphysician expertise"
)

add_box(
    8.4, 3.7, 2.9, 1.3,
    "Symptom–specialty\nalignment score\nCosine similarity"
)

add_box(
    8.4, 1.4, 2.9, 1.2,
    "Patient satisfaction outcome\nBelow-5-star rating"
)

add_box(
    4.4, 0.7, 3.2, 1.2,
    "Control variables\nReview count, patient count,\nreview type, province, text length"
)

# Arrows
add_arrow(3.5, 5.25, 4.4, 4.55)
add_arrow(3.5, 3.35, 4.4, 4.15)
add_arrow(7.6, 4.35, 8.4, 4.35)
add_arrow(9.85, 3.7, 9.85, 2.6)
add_arrow(7.6, 1.3, 8.4, 1.9)

# Revised title
ax.text(
    6, 6.45,
    "Figure 1. Conceptual framework for LLM-derived symptom–specialty alignment and patient satisfaction",
    ha="center",
    va="center",
    fontsize=14,
    fontweight="bold"
)

# Revised note
ax.text(
    6, 0.2,
    "Note. The framework links patient-expressed needs and physician expertise through LLM-derived semantic alignment.",
    ha="center",
    va="center",
    fontsize=10
)

plt.tight_layout()

save_path = figures_dir / "Figure1_conceptual_framework_revised.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", save_path)

In [ ]:
from pathlib import Path

# Project directory
project_dir = Path("/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit")

# Output folders
tables_dir = project_dir / "outputs" / "tables"
figures_dir = project_dir / "outputs" / "figures"

tables_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

print("Folders created successfully.")

In [ ]:
# Recreate clean Table 4
var_labels = {
    "alignment_llm_z": "LLM alignment",
    "alignment_tfidf_z": "TF-IDF alignment",
    "log_review_count": "Log review count",
    "log_patient_count": "Log patient count",
    "patient_need_length_main": "Patient text length"
}

key_vars = list(var_labels.keys())

def stars(p):
    if pd.isna(p):
        return ""
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return ""

def format_or_ci(or_value, ci_low, ci_high, p_value):
    if pd.isna(or_value):
        return ""
    return f"{or_value:.3f}{stars(p_value)} [{ci_low:.3f}, {ci_high:.3f}]"

clean_rows = []

for var in key_vars:
    row = {"Variable": var_labels[var]}
    
    for model in ["M1", "M2", "M3", "M4"]:
        match = table4[table4["variable"] == var]
        
        if len(match) == 0:
            row[model] = ""
        else:
            match = match.iloc[0]
            row[model] = format_or_ci(
                match.get(f"{model}_OR"),
                match.get(f"{model}_CI_low"),
                match.get(f"{model}_CI_high"),
                match.get(f"{model}_p")
            )
    
    clean_rows.append(row)

table4_clean = pd.DataFrame(clean_rows)

table4_clean = pd.concat([
    table4_clean,
    pd.DataFrame([
        {"Variable": "Platform controls", "M1": "No", "M2": "Yes", "M3": "Yes", "M4": "Yes"},
        {"Variable": "Review type controls", "M1": "No", "M2": "No", "M3": "Yes", "M4": "Yes"},
        {"Variable": "Province controls", "M1": "No", "M2": "No", "M3": "Yes", "M4": "Yes"},
        {"Variable": "N", "M1": int(model1.nobs), "M2": int(model2.nobs), "M3": int(model3.nobs), "M4": int(model4.nobs)},
        {"Variable": "Pseudo R²", "M1": f"{model1.prsquared:.3f}", "M2": f"{model2.prsquared:.3f}", "M3": f"{model3.prsquared:.3f}", "M4": f"{model4.prsquared:.3f}"}
    ])
], ignore_index=True)

table4_clean

In [ ]:
table4_clean.to_csv(tables_dir / "Table4_regression_models.csv", index=False)
table4_clean.to_excel(tables_dir / "Table4_regression_models.xlsx", index=False)

print("Table 4 exported successfully.")

In [ ]:
plt.savefig(
    figures_dir / "Figure1_conceptual_framework.png",
    dpi=300,
    bbox_inches="tight"
)

In [ ]:
plt.savefig(
    figures_dir / "Figure2_llm_alignment_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

In [ ]:
plt.savefig(
    figures_dir / "Figure3_alignment_boxplot.png",
    dpi=300,
    bbox_inches="tight"
)

In [ ]:
plt.savefig(
    figures_dir / "Figure4_predicted_probability.png",
    dpi=300,
    bbox_inches="tight"
)

In [ ]:
import numpy as np
import pandas as pd

table5 = pd.DataFrame({
    "Model": ["M1", "M2", "M3", "M4"],
    "Coefficient": [
        model1.params["alignment_llm_z"],
        model2.params["alignment_llm_z"],
        model3.params["alignment_llm_z"],
        model4.params["alignment_llm_z"]
    ],
    "Odds ratio": [
        np.exp(model1.params["alignment_llm_z"]),
        np.exp(model2.params["alignment_llm_z"]),
        np.exp(model3.params["alignment_llm_z"]),
        np.exp(model4.params["alignment_llm_z"])
    ],
    "Percent change in odds": [
        (np.exp(model1.params["alignment_llm_z"]) - 1) * 100,
        (np.exp(model2.params["alignment_llm_z"]) - 1) * 100,
        (np.exp(model3.params["alignment_llm_z"]) - 1) * 100,
        (np.exp(model4.params["alignment_llm_z"]) - 1) * 100
    ],
    "p-value": [
        model1.pvalues["alignment_llm_z"],
        model2.pvalues["alignment_llm_z"],
        model3.pvalues["alignment_llm_z"],
        model4.pvalues["alignment_llm_z"]
    ]
})

table5

In [ ]:
with pd.ExcelWriter(
    tables_dir / "Paper4_Tables.xlsx",
    engine="openpyxl"
) as writer:

    table1.to_excel(writer, sheet_name="Table1", index=True)
    table2.to_excel(writer, sheet_name="Table2", index=True)
    table3.to_excel(writer, sheet_name="Table3", index=False)
    table4_clean.to_excel(writer, sheet_name="Table4", index=False)
    table5.to_excel(writer, sheet_name="Table5", index=False)

print("Excel tables exported.")

In [ ]:
df.to_csv(
    project_dir / "outputs" / "final_analysis_dataset.csv",
    index=False
)

print("Final analysis dataset saved.")

In [ ]:
readme_text = """
Paper 4: LLM-Derived Patient-Physician Alignment

Folder structure:

outputs/
    tables/
    figures/

Tables:
- Table1_descriptive_statistics.csv
- Table2_correlation_matrix.csv
- Table3_vif_results.csv
- Table4_regression_models.csv
- Table5_main_effect_summary.csv

Figures:
- Figure1_conceptual_framework.png
- Figure2_llm_alignment_distribution.png
- Figure3_alignment_boxplot.png
- Figure4_predicted_probability.png

Dataset:
- final_analysis_dataset.csv

Main dependent variable:
lower_rating
(1 = below-5-star rating, 0 = five-star rating)

Main predictor:
alignment_llm
"""

with open(project_dir / "outputs" / "README.txt", "w") as f:
    f.write(readme_text)

print("README created.")

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

robust_df = df.copy()

# Alternative stricter dissatisfaction outcome
robust_df["low_rating_3"] = np.where(robust_df["rating"] <= 3, 1, 0)

print(robust_df["low_rating_3"].value_counts())
print(round(robust_df["low_rating_3"].value_counts(normalize=True) * 100, 2))

In [ ]:
robust_model_low3 = smf.logit(
    formula="""
    low_rating_3 ~ alignment_llm_z
    + log_review_count
    + log_patient_count
    + patient_need_length_main
    + C(评论类型)
    + C(province_group)
    """,
    data=robust_df
).fit(method="lbfgs", maxiter=500)

print(robust_model_low3.summary())

In [ ]:
coef = robust_model_low3.params["alignment_llm_z"]
or_value = np.exp(coef)

print("Coefficient:", coef)
print("Odds ratio:", or_value)
print("Percent change in odds:", round((or_value - 1) * 100, 2), "%")
print("p-value:", robust_model_low3.pvalues["alignment_llm_z"])

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

project_dir = Path("/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit")

random_df = pd.read_csv(
    project_dir / "data" / "samples" / "paper4_sample_50000_random.csv",
    encoding="utf-8-sig"
)

print(random_df.shape)
print(random_df["lower_rating"].value_counts())
print(round(random_df["lower_rating"].value_counts(normalize=True) * 100, 2))

In [ ]:
robust_model_llm_only = smf.logit(
    formula="""
    lower_rating ~ alignment_llm_z
    + log_review_count
    + log_patient_count
    + patient_need_length_main
    + C(评论类型)
    + C(province_group)
    """,
    data=df
).fit(method="lbfgs", maxiter=500)

print(robust_model_llm_only.summary())

coef = robust_model_llm_only.params["alignment_llm_z"]
or_value = np.exp(coef)

print("Coefficient:", coef)
print("Odds ratio:", or_value)
print("Percent change in odds:", round((or_value - 1) * 100, 2), "%")
print("p-value:", robust_model_llm_only.pvalues["alignment_llm_z"])

In [ ]:
import pandas as pd
import numpy as np

robustness_table = pd.DataFrame({

    "Robustness check": [
        "Main model (LLM + controls)",
        "Alternative outcome (rating ≤ 3)",
        "LLM-only model"
    ],

    "Coefficient": [
        -0.304048,
        -0.402783,
        -0.269335
    ],

    "Odds ratio": [
        0.737825,
        0.668457,
        0.763887
    ],

    "Percent change in odds": [
        -26.22,
        -33.15,
        -23.61
    ],

    "p-value": [
        "p < .001",
        "p < .001",
        "p < .001"
    ]
})

print(robustness_table.round(3))

In [ ]:
robustness_table.to_csv(
    tables_dir / "Table6_robustness_checks.csv",
    index=False
)

print("Robustness table exported.")

In [ ]:
import pandas as pd
from pathlib import Path

project_dir = Path("/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit")

df = pd.read_csv(
    project_dir / "data" / "processed" / "paper4_model_dataset_with_tfidf_llm_alignment.csv",
    encoding="utf-8-sig"
)

# 1. Confirm final sample size
print("1. Final sample size")
print("N =", len(df))

# 2. Confirm outcome variable definition
print("\n2. Outcome variable check")
print(df["rating"].value_counts().sort_index())
print("\nlower_rating distribution:")
print(df["lower_rating"].value_counts().sort_index())

check_outcome = (
    ((df["rating"] < 5) & (df["lower_rating"] == 1)) |
    ((df["rating"] == 5) & (df["lower_rating"] == 0))
).all()

print("\nOutcome correctly coded:", check_outcome)

# 3. Confirm LLM model used
llm_model_name = "paraphrase-multilingual-MiniLM-L12-v2"

print("\n3. LLM model used")
print(llm_model_name)

# Optional: confirm alignment column exists
print("\nAlignment columns available:")
print([col for col in df.columns if "alignment" in col])

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

project_dir = Path("/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit")

df = pd.read_csv(
    project_dir / "data" / "processed" / "paper4_model_dataset_with_tfidf_llm_alignment.csv",
    encoding="utf-8-sig"
)

coding_dir = project_dir / "outputs" / "human_validation"
coding_dir.mkdir(parents=True, exist_ok=True)

# Create high and low alignment groups
df["alignment_group"] = pd.qcut(
    df["alignment_llm"],
    q=4,
    labels=["Low", "Medium-low", "Medium-high", "High"]
)

low_cases = df[df["alignment_group"] == "Low"].sample(n=100, random_state=42)
high_cases = df[df["alignment_group"] == "High"].sample(n=100, random_state=42)

validation_sample = pd.concat([low_cases, high_cases], axis=0)
validation_sample = validation_sample.sample(frac=1, random_state=42).reset_index(drop=True)

# Create anonymous validation ID
validation_sample["validation_id"] = ["VAL_" + str(i).zfill(4) for i in range(1, len(validation_sample) + 1)]

# Fields for coders
coder_sheet = validation_sample[[
    "validation_id",
    "patient_need_text_main",
    "physician_expertise_text_main"
]].copy()

# Add empty coding columns
coder_sheet["alignment_code"] = ""
coder_sheet["coder_confidence"] = ""
coder_sheet["notes"] = ""

# Save two identical sheets
coder1_path = coding_dir / "paper4_validation_coder1.xlsx"
coder2_path = coding_dir / "paper4_validation_coder2.xlsx"

coder_sheet.to_excel(coder1_path, index=False)
coder_sheet.to_excel(coder2_path, index=False)

# Save master file with hidden true scores for later comparison
master_sheet = validation_sample[[
    "validation_id",
    "alignment_group",
    "alignment_llm",
    "alignment_tfidf",
    "lower_rating",
    "rating",
    "patient_need_text_main",
    "physician_expertise_text_main"
]].copy()

master_path = coding_dir / "paper4_validation_master_with_scores.xlsx"
master_sheet.to_excel(master_path, index=False)

print("Saved:")
print(coder1_path)
print(coder2_path)
print(master_path)

In [ ]:
import os

for file in os.listdir():
    if file.endswith(".csv"):
        print(file)

In [ ]:
import os

for root, dirs, files in os.walk("."):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

In [ ]:
import os

project_folder = "/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit"

os.chdir(project_folder)

print("Current directory:")
print(os.getcwd())

In [ ]:
import os
import pandas as pd

required_columns = {
    "alignment_llm_z",
    "alignment_tfidf_z",
    "lower_rating"
}

for file in os.listdir(project_folder):

    if file.endswith(".csv"):

        try:
            temp = pd.read_csv(file, nrows=5)

            if required_columns.issubset(temp.columns):

                print("="*60)
                print("FOUND FINAL DATASET")
                print("Filename:", file)

                df = pd.read_csv(file)

                print("Shape:", df.shape)

                print("Columns:")
                print(df.columns.tolist())

        except Exception as e:
            pass

In [ ]:
df.shape

In [ ]:
import os

print(os.getcwd())

In [ ]:
import os

for file in sorted(os.listdir()):
    print(file)

In [ ]:
import os

data_folder = "/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit/data"

for file in sorted(os.listdir(data_folder)):
    print(file)

In [ ]:
import os

processed_folder = "/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit/data/processed"

for file in sorted(os.listdir(processed_folder)):
    print(file)

In [ ]:
import pandas as pd
import os

processed_folder = "/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit/data/processed"

df = pd.read_csv(
    os.path.join(processed_folder,
                 "paper4_final_regression_dataset.csv")
)

print(df.shape)

print("\nColumns:\n")
print(df.columns.tolist())

In [ ]:
import pandas as pd

# Variables to report
variables = [
    "lower_rating",
    "alignment_llm_z",
    "alignment_tfidf_z",
    "log_review_count",
    "log_patient_count",
    "patient_need_length_main"
]

# Friendly names
rename = {
    "lower_rating": "Reduced patient satisfaction",
    "alignment_llm_z": "Embedding-based semantic alignment",
    "alignment_tfidf_z": "TF-IDF lexical alignment",
    "log_review_count": "Log review count",
    "log_patient_count": "Log patient count",
    "patient_need_length_main": "Patient need text length"
}

# Variable descriptions
descriptions = {
    "Reduced patient satisfaction":
        "Binary outcome (1 = rating < 5; 0 = rating = 5)",

    "Embedding-based semantic alignment":
        "Standardized cosine similarity between multilingual transformer-based sentence embeddings",

    "TF-IDF lexical alignment":
        "Standardized cosine similarity between TF-IDF representations",

    "Log review count":
        "Natural logarithm of physician review count",

    "Log patient count":
        "Natural logarithm of physician patient count",

    "Patient need text length":
        "Length (characters) of processed patient-side text"
}

# Descriptive statistics
table1 = (
    df[variables]
    .describe()
    .T[["mean", "std", "min", "max"]]
)

table1.index = [rename[v] for v in table1.index]

table1.insert(
    0,
    "Description",
    [descriptions[i] for i in table1.index]
)

table1 = table1.rename(columns={
    "mean":"Mean",
    "std":"SD",
    "min":"Min",
    "max":"Max"
})

table1 = table1.round(3)

print(table1)

table1.to_excel(
    "Table1_Descriptive_Statistics.xlsx"
)

In [ ]:
import os

# Create folder if it does not already exist
output_folder = "/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit/outputs/tables"

os.makedirs(output_folder, exist_ok=True)

# Save as Excel
table1.to_excel(
    os.path.join(output_folder, "Table1_Descriptive_Statistics.xlsx"),
    index=True
)

print("Table 1 saved successfully.")

In [ ]:
table1.to_csv(
    os.path.join(output_folder, "Table1_Descriptive_Statistics.csv"),
    index=True,
    encoding="utf-8-sig"
)

In [ ]:
df["alignment_tfidf_z"].describe()

In [ ]:
df["alignment_tfidf_z"].nlargest(10)

In [ ]:
df["alignment_tfidf"].describe()

In [ ]:
df["alignment_llm"].describe()

In [ ]:
import pandas as pd
import os

# ==========================================================
# Variables to include (RAW alignment variables)
# ==========================================================

variables = [
    "lower_rating",
    "alignment_llm",
    "alignment_tfidf",
    "log_review_count",
    "log_patient_count",
    "patient_need_length_main"
]

# ==========================================================
# Friendly variable names
# ==========================================================

rename = {
    "lower_rating": "Reduced patient satisfaction",
    "alignment_llm": "Embedding-based semantic alignment",
    "alignment_tfidf": "TF-IDF lexical alignment",
    "log_review_count": "Log review count",
    "log_patient_count": "Log patient count",
    "patient_need_length_main": "Patient need text length"
}

# ==========================================================
# Variable descriptions
# ==========================================================

descriptions = {
    "Reduced patient satisfaction":
        "Binary outcome (1 = rating < 5; 0 = rating = 5)",

    "Embedding-based semantic alignment":
        "Cosine similarity between transformer-based sentence embeddings",

    "TF-IDF lexical alignment":
        "Cosine similarity between TF-IDF text representations",

    "Log review count":
        "Natural logarithm of physician review count",

    "Log patient count":
        "Natural logarithm of physician patient count",

    "Patient need text length":
        "Length (characters) of processed patient-side text"
}

# ==========================================================
# Create descriptive statistics
# ==========================================================

table1 = df[variables].describe().T

table1 = table1[["mean", "std", "50%", "min", "max"]]

table1.columns = [
    "Mean",
    "SD",
    "Median",
    "Min",
    "Max"
]

table1.index = [rename[v] for v in table1.index]

table1.insert(
    0,
    "Description",
    [descriptions[v] for v in table1.index]
)

table1 = table1.round(3)

table1.index.name = "Variable"

# ==========================================================
# Save table
# ==========================================================

output_folder = "/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit/outputs/tables"

os.makedirs(output_folder, exist_ok=True)

table1.to_excel(
    os.path.join(
        output_folder,
        "Table1_Descriptive_Statistics.xlsx"
    )
)

table1.to_csv(
    os.path.join(
        output_folder,
        "Table1_Descriptive_Statistics.csv"
    ),
    encoding="utf-8-sig"
)

print(table1)

print("\nTable 1 saved successfully.")

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr
import os

# ==========================================================
# Variables (RAW variables used in descriptive analyses)
# ==========================================================

variables = [
    "lower_rating",
    "alignment_llm",
    "alignment_tfidf",
    "log_review_count",
    "log_patient_count",
    "patient_need_length_main"
]

# ==========================================================
# Friendly names
# ==========================================================

names = {
    "lower_rating": "1. Reduced patient satisfaction",
    "alignment_llm": "2. Embedding-based semantic alignment",
    "alignment_tfidf": "3. TF-IDF lexical alignment",
    "log_review_count": "4. Log review count",
    "log_patient_count": "5. Log patient count",
    "patient_need_length_main": "6. Patient need text length"
}

# ==========================================================
# Function to add significance stars
# ==========================================================

def corr_sig(x, y):

    r, p = pearsonr(x, y)

    if p < 0.001:
        star = "***"
    elif p < 0.01:
        star = "**"
    elif p < 0.05:
        star = "*"
    else:
        star = ""

    return f"{r:.3f}{star}"

# ==========================================================
# Build lower-triangular correlation matrix
# ==========================================================

corr_table = pd.DataFrame(
    "",
    index=[names[v] for v in variables],
    columns=[str(i) for i in range(1, len(variables)+1)]
)

for i, var1 in enumerate(variables):

    for j, var2 in enumerate(variables):

        if i == j:
            corr_table.iloc[i, j] = "1.000"

        elif i > j:

            corr_table.iloc[i, j] = corr_sig(
                df[var1],
                df[var2]
            )

# ==========================================================
# Save
# ==========================================================

output_folder = "/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit/outputs/tables"

os.makedirs(output_folder, exist_ok=True)

corr_table.to_excel(
    os.path.join(
        output_folder,
        "Table2_Correlation_Matrix.xlsx"
    )
)

corr_table.to_csv(
    os.path.join(
        output_folder,
        "Table2_Correlation_Matrix.csv"
    ),
    encoding="utf-8-sig"
)

print(corr_table)

print("\nTable 2 saved successfully.")

In [ ]:
import pandas as pd
import numpy as np
import os
from math import erf, sqrt

# Variables
variables = [
    "lower_rating",
    "alignment_llm",
    "alignment_tfidf",
    "log_review_count",
    "log_patient_count",
    "patient_need_length_main"
]

names = {
    "lower_rating": "1. Reduced patient satisfaction",
    "alignment_llm": "2. Embedding-based semantic alignment",
    "alignment_tfidf": "3. TF-IDF lexical alignment",
    "log_review_count": "4. Log review count",
    "log_patient_count": "5. Log patient count",
    "patient_need_length_main": "6. Patient need text length"
}

# Approximate p-value without scipy
def norm_cdf(z):
    return 0.5 * (1 + erf(z / sqrt(2)))

def corr_with_sig(x, y):
    x = pd.Series(x).dropna()
    y = pd.Series(y).dropna()
    common = x.index.intersection(y.index)
    x = x.loc[common]
    y = y.loc[common]

    r = np.corrcoef(x, y)[0, 1]
    n = len(x)

    # large-sample approximation for p-value
    z = abs(r) * np.sqrt(n - 3)
    p = 2 * (1 - norm_cdf(z))

    if p < 0.001:
        star = "***"
    elif p < 0.01:
        star = "**"
    elif p < 0.05:
        star = "*"
    else:
        star = ""

    return f"{r:.3f}{star}"

# Build lower-triangle matrix
table2 = pd.DataFrame(
    "",
    index=[names[v] for v in variables],
    columns=[str(i) for i in range(1, len(variables) + 1)]
)

for i, var1 in enumerate(variables):
    for j, var2 in enumerate(variables):
        if i == j:
            table2.iloc[i, j] = "1.000"
        elif i > j:
            table2.iloc[i, j] = corr_with_sig(df[var1], df[var2])

table2.index.name = "Variable"

# Save
output_folder = "/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit/outputs/tables"
os.makedirs(output_folder, exist_ok=True)

table2.to_excel(os.path.join(output_folder, "Table2_Correlation_Matrix.xlsx"))
table2.to_csv(
    os.path.join(output_folder, "Table2_Correlation_Matrix.csv"),
    encoding="utf-8-sig"
)

print(table2)
print("\nTable 2 saved successfully.")

In [ ]:
import pandas as pd
import os
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ==========================================================
# Variables used in the regression
# ==========================================================

vif_vars = [
    "alignment_llm_z",
    "alignment_tfidf_z",
    "log_review_count",
    "log_patient_count",
    "patient_need_length_main"
]

X = df[vif_vars].copy()

# ==========================================================
# Calculate VIF
# ==========================================================

vif_table = pd.DataFrame()

vif_table["Variable"] = [
    "Embedding-based semantic alignment",
    "TF-IDF lexical alignment",
    "Log review count",
    "Log patient count",
    "Patient need text length"
]

vif_table["VIF"] = [
    variance_inflation_factor(X.values, i)
    for i in range(X.shape[1])
]

vif_table["Tolerance"] = 1 / vif_table["VIF"]

vif_table = vif_table.round(3)

print(vif_table)

# ==========================================================
# Save
# ==========================================================

output_folder = "/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit/outputs/tables"

os.makedirs(output_folder, exist_ok=True)

vif_table.to_excel(
    os.path.join(
        output_folder,
        "Table3_VIF.xlsx"
    ),
    index=False
)

vif_table.to_csv(
    os.path.join(
        output_folder,
        "Table3_VIF.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

print("\nTable 3 saved successfully.")

In [ ]:
import pandas as pd
import numpy as np
import os

# Variables used for VIF
vif_vars = [
    "alignment_llm_z",
    "alignment_tfidf_z",
    "log_review_count",
    "log_patient_count",
    "patient_need_length_main"
]

display_names = {
    "alignment_llm_z": "Embedding-based semantic alignment",
    "alignment_tfidf_z": "TF-IDF lexical alignment",
    "log_review_count": "Log review count",
    "log_patient_count": "Log patient count",
    "patient_need_length_main": "Patient need text length"
}

X = df[vif_vars].dropna().copy()

def calculate_vif_numpy(data, target_col):
    y = data[target_col].values
    X_other = data.drop(columns=[target_col]).values

    # Add intercept
    X_other = np.column_stack([np.ones(X_other.shape[0]), X_other])

    # OLS using numpy only
    beta = np.linalg.lstsq(X_other, y, rcond=None)[0]
    y_hat = X_other @ beta

    ss_res = np.sum((y - y_hat) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)

    r_squared = 1 - (ss_res / ss_tot)

    vif = 1 / (1 - r_squared)
    tolerance = 1 / vif

    return vif, tolerance

rows = []

for col in vif_vars:
    vif, tolerance = calculate_vif_numpy(X, col)

    rows.append({
        "Variable": display_names[col],
        "VIF": vif,
        "Tolerance": tolerance
    })

table3 = pd.DataFrame(rows).round(3)

print(table3)

# Save
output_folder = "/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit/outputs/tables"
os.makedirs(output_folder, exist_ok=True)

table3.to_excel(
    os.path.join(output_folder, "Table3_VIF.xlsx"),
    index=False
)

table3.to_csv(
    os.path.join(output_folder, "Table3_VIF.csv"),
    index=False,
    encoding="utf-8-sig"
)

print("\nTable 3 saved successfully.")

In [ ]:
import sys
import numpy
import scipy
import statsmodels
import sklearn

print("Python executable:")
print(sys.executable)

print("\nEnvironment:")
print(sys.prefix)

print("\nVersions:")
print("NumPy:", numpy.__version__)
print("SciPy:", scipy.__version__)
print("Statsmodels:", statsmodels.__version__)
print("Scikit-learn:", sklearn.__version__)

In [ ]:
import statsmodels.api as sm
import scipy
import numpy as np

print("Statsmodels OK!")

In [ ]:
import sentence_transformers
import transformers
import torch
import accelerate
import openai

print("sentence-transformers:", sentence_transformers.__version__)
print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("accelerate:", accelerate.__version__)
print("openai:", openai.__version__)

print("\nAll NLP packages are installed correctly!")

In [ ]:
import pandas as pd

df = pd.read_csv(
    "Paper4_Patient_Physician_Fit/data/processed/paper4_final_regression_dataset.csv"
)

print(df.shape)
df.head()

In [ ]:
import os

print(os.getcwd())

In [ ]:
import os

for root, dirs, files in os.walk("/Users/isaacasante"):
    if "paper4_final_regression_dataset.csv" in files:
        print(os.path.join(root, "paper4_final_regression_dataset.csv"))

In [ ]:
import pandas as pd

df = pd.read_csv(
    "/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit/data/processed/paper4_final_regression_dataset.csv"
)

print(df.shape)
df.head()

In [ ]:
print(df.columns.tolist())

In [ ]:
df.columns

In [ ]:
import pandas as pd
import numpy as np
import os
import statsmodels.formula.api as smf

# Fit models
m1 = smf.logit("lower_rating ~ alignment_llm_z", data=df).fit()

m2 = smf.logit(
    "lower_rating ~ alignment_llm_z + log_review_count + log_patient_count + patient_need_length_main",
    data=df
).fit()

m3 = smf.logit(
    "lower_rating ~ alignment_llm_z + log_review_count + log_patient_count + patient_need_length_main + C(评论类型) + C(province_group)",
    data=df
).fit()

m4 = smf.logit(
    "lower_rating ~ alignment_llm_z + alignment_tfidf_z + log_review_count + log_patient_count + patient_need_length_main + C(评论类型) + C(province_group)",
    data=df
).fit()

models = [m1, m2, m3, m4]
model_names = ["Model 1", "Model 2", "Model 3", "Model 4"]

# Display names
display_names = {
    "alignment_llm_z": "Embedding-based semantic alignment",
    "alignment_tfidf_z": "TF-IDF lexical alignment",
    "log_review_count": "Log review count",
    "log_patient_count": "Log patient count",
    "patient_need_length_main": "Patient need text length"
}

main_vars = [
    "alignment_llm_z",
    "alignment_tfidf_z",
    "log_review_count",
    "log_patient_count",
    "patient_need_length_main"
]

def stars(p):
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return ""

def format_or_ci(model, var):
    if var not in model.params.index:
        return ""
    OR = np.exp(model.params[var])
    ci = model.conf_int().loc[var]
    lower = np.exp(ci[0])
    upper = np.exp(ci[1])
    p = model.pvalues[var]
    return f"{OR:.3f}{stars(p)} [{lower:.3f}, {upper:.3f}]"

# Build table
rows = []

for var in main_vars:
    row = {"Variable": display_names[var]}
    for name, model in zip(model_names, models):
        row[name] = format_or_ci(model, var)
    rows.append(row)

# Fixed effects rows
rows.append({
    "Variable": "Consultation type fixed effects",
    "Model 1": "No",
    "Model 2": "No",
    "Model 3": "Yes",
    "Model 4": "Yes"
})

rows.append({
    "Variable": "Province fixed effects",
    "Model 1": "No",
    "Model 2": "No",
    "Model 3": "Yes",
    "Model 4": "Yes"
})

# Model statistics
rows.append({
    "Variable": "Observations",
    "Model 1": int(m1.nobs),
    "Model 2": int(m2.nobs),
    "Model 3": int(m3.nobs),
    "Model 4": int(m4.nobs)
})

rows.append({
    "Variable": "McFadden pseudo-R²",
    "Model 1": f"{m1.prsquared:.3f}",
    "Model 2": f"{m2.prsquared:.3f}",
    "Model 3": f"{m3.prsquared:.3f}",
    "Model 4": f"{m4.prsquared:.3f}"
})

rows.append({
    "Variable": "Log-likelihood",
    "Model 1": f"{m1.llf:.3f}",
    "Model 2": f"{m2.llf:.3f}",
    "Model 3": f"{m3.llf:.3f}",
    "Model 4": f"{m4.llf:.3f}"
})

rows.append({
    "Variable": "AIC",
    "Model 1": f"{m1.aic:.3f}",
    "Model 2": f"{m2.aic:.3f}",
    "Model 3": f"{m3.aic:.3f}",
    "Model 4": f"{m4.aic:.3f}"
})

table4 = pd.DataFrame(rows)

# Save
output_folder = "/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit/outputs/tables"
os.makedirs(output_folder, exist_ok=True)

table4.to_excel(
    os.path.join(output_folder, "Table4_Logistic_Regression.xlsx"),
    index=False
)

table4.to_csv(
    os.path.join(output_folder, "Table4_Logistic_Regression.csv"),
    index=False,
    encoding="utf-8-sig"
)

print(table4)
print("\nTable 4 saved successfully.")

In [ ]:
import pandas as pd

# Full coefficient table for Model 4
coef = m4.summary2().tables[1]

coef["Odds Ratio"] = coef["Coef."].apply(np.exp)
coef["CI Lower"] = coef["[0.025"].apply(np.exp)
coef["CI Upper"] = coef["0.975]"].apply(np.exp)

supp_table = coef[
    ["Odds Ratio", "Std.Err.", "z", "P>|z|", "CI Lower", "CI Upper"]
].round(4)

supp_table.to_excel(
    "/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit/outputs/tables/Supplementary_Table_S1_Model4.xlsx"
)

print(supp_table)

In [ ]:
# ==========================================================
# Robustness Analysis 1
# Random Forest Classifier
# ==========================================================

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
)

features = [
    "alignment_llm_z",
    "alignment_tfidf_z",
    "log_review_count",
    "log_patient_count",
    "patient_need_length_main",
]

X = pd.get_dummies(
    df[
        features +
        ["评论类型", "province_group"]
    ],
    drop_first=True
)

y = df["lower_rating"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

rf = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

pred = rf.predict(X_test)
prob = rf.predict_proba(X_test)[:,1]

print("Accuracy :", accuracy_score(y_test,pred))
print("Precision:", precision_score(y_test,pred))
print("Recall   :", recall_score(y_test,pred))
print("F1       :", f1_score(y_test,pred))
print("ROC AUC  :", roc_auc_score(y_test,prob))

print("\n")
print(classification_report(y_test,pred))

In [ ]:
importance = (
    pd.DataFrame({
        "Variable":X.columns,
        "Importance":rf.feature_importances_
    })
    .sort_values("Importance",ascending=False)
)

print(importance.head(20))

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

output_tables = "/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit/outputs/tables"
output_figures = "/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit/outputs/figures"

os.makedirs(output_tables, exist_ok=True)
os.makedirs(output_figures, exist_ok=True)

# -----------------------------
# Table S2: Random Forest performance
# -----------------------------

rf_performance = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"],
    "Value": [
        accuracy_score(y_test, pred),
        precision_score(y_test, pred),
        recall_score(y_test, pred),
        f1_score(y_test, pred),
        roc_auc_score(y_test, prob)
    ]
})

rf_performance["Value"] = rf_performance["Value"].round(3)

rf_performance.to_excel(
    os.path.join(output_tables, "TableS2_Random_Forest_Performance.xlsx"),
    index=False
)

display(rf_performance)

# -----------------------------
# Table S3: Random Forest feature importance
# -----------------------------

feature_names = {
    "alignment_llm_z": "Embedding-based semantic alignment",
    "alignment_tfidf_z": "TF-IDF lexical alignment",
    "log_review_count": "Log review count",
    "log_patient_count": "Log patient count",
    "patient_need_length_main": "Patient need text length"
}

rf_importance = importance.copy()
rf_importance["Variable"] = rf_importance["Variable"].replace(feature_names)
rf_importance.insert(0, "Rank", range(1, len(rf_importance) + 1))
rf_importance["Importance"] = rf_importance["Importance"].round(3)

rf_importance.to_excel(
    os.path.join(output_tables, "TableS3_Random_Forest_Feature_Importance.xlsx"),
    index=False
)

display(rf_importance.head(20))

# -----------------------------
# Figure S1: Feature importance plot
# -----------------------------

plot_data = rf_importance.head(10).sort_values("Importance")

plt.figure(figsize=(8, 5))
plt.barh(plot_data["Variable"], plot_data["Importance"])
plt.xlabel("Feature importance")
plt.ylabel("")
plt.title("Random Forest feature importance")
plt.tight_layout()

plt.savefig(
    os.path.join(output_figures, "FigureS1_Random_Forest_Feature_Importance.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# -----------------------------
# Figure S2: ROC curve
# -----------------------------

fpr, tpr, thresholds = roc_curve(y_test, prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"ROC-AUC = {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("Random Forest ROC curve")
plt.legend(loc="lower right")
plt.tight_layout()

plt.savefig(
    os.path.join(output_figures, "FigureS2_Random_Forest_ROC_Curve.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Random Forest tables and figures saved successfully.")

In [ ]:
import pandas as pd
import os

output_tables = "/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit/outputs/tables"

# -------------------------
# Panel A
# -------------------------

panelA = pd.DataFrame({
    "Metric":[
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "ROC-AUC"
    ],
    "Value":[
        round(accuracy_score(y_test, pred),3),
        round(precision_score(y_test, pred),3),
        round(recall_score(y_test, pred),3),
        round(f1_score(y_test, pred),3),
        round(roc_auc_score(y_test, prob),3)
    ]
})

# -------------------------
# Panel B
# -------------------------

top5 = rf_importance[
    rf_importance["Variable"].isin([
        "Embedding-based semantic alignment",
        "Patient need text length",
        "Log review count",
        "Log patient count",
        "TF-IDF lexical alignment"
    ])
].copy()

top5 = top5.sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

top5.insert(0,"Rank",range(1,len(top5)+1))

# -------------------------
# Save into ONE Excel file
# -------------------------

outfile = os.path.join(
    output_tables,
    "Table5_Random_Forest_Robustness.xlsx"
)

with pd.ExcelWriter(outfile, engine="openpyxl") as writer:

    panelA.to_excel(
        writer,
        sheet_name="Table5",
        startrow=1,
        index=False
    )

    top5.to_excel(
        writer,
        sheet_name="Table5",
        startrow=10,
        index=False
    )

print("Table 5 saved successfully!")
print(outfile)

display(panelA)
display(top5)

In [ ]:
import pandas as pd
import os
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

# =====================================================
# Output folder
# =====================================================

output_tables = "/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit/outputs/tables"

os.makedirs(output_tables, exist_ok=True)

# =====================================================
# Panel A: Model Performance
# =====================================================

panelA = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "ROC-AUC"
    ],
    "Value": [
        round(accuracy_score(y_test, pred), 3),
        round(precision_score(y_test, pred), 3),
        round(recall_score(y_test, pred), 3),
        round(f1_score(y_test, pred), 3),
        round(roc_auc_score(y_test, prob), 3)
    ]
})

# =====================================================
# Panel B: Top 5 substantive predictors only
# =====================================================

feature_map = {
    "alignment_llm_z": "Embedding-based semantic alignment",
    "patient_need_length_main": "Patient need text length",
    "log_review_count": "Log review count",
    "log_patient_count": "Log patient count",
    "alignment_tfidf_z": "TF-IDF lexical alignment"
}

# Build a completely new dataframe (ignores all dummy variables)
panelB = pd.DataFrame({
    "Predictor": [
        feature_map["alignment_llm_z"],
        feature_map["patient_need_length_main"],
        feature_map["log_review_count"],
        feature_map["log_patient_count"],
        feature_map["alignment_tfidf_z"]
    ],
    "Importance": [
        rf.feature_importances_[X.columns.get_loc("alignment_llm_z")],
        rf.feature_importances_[X.columns.get_loc("patient_need_length_main")],
        rf.feature_importances_[X.columns.get_loc("log_review_count")],
        rf.feature_importances_[X.columns.get_loc("log_patient_count")],
        rf.feature_importances_[X.columns.get_loc("alignment_tfidf_z")]
    ]
})

panelB = panelB.sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

panelB.insert(0, "Rank", range(1, len(panelB) + 1))

panelB["Importance"] = panelB["Importance"].round(3)

# =====================================================
# Save as ONE publication-ready table
# =====================================================

outfile = os.path.join(
    output_tables,
    "Table5_Random_Forest_Robustness.xlsx"
)

with pd.ExcelWriter(outfile, engine="openpyxl") as writer:

    panelA.to_excel(
        writer,
        sheet_name="Table5",
        startrow=1,
        index=False
    )

    panelB.to_excel(
        writer,
        sheet_name="Table5",
        startrow=10,
        index=False
    )

print("===================================")
print("Table 5 saved successfully.")
print(outfile)
print("===================================")

print("\nPanel A: Model Performance")
display(panelA)

print("\nPanel B: Top Predictors")
display(panelB)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
import pandas as pd

# -----------------------------------------
# Four feature sets
# -----------------------------------------

feature_sets = {

    "RF-1 Controls only":
    [
        "log_review_count",
        "log_patient_count",
        "patient_need_length_main"
    ] + [c for c in X.columns if c.startswith("评论类型_")] \
      + [c for c in X.columns if c.startswith("province_group_")],

    "RF-2 Controls + TF-IDF":
    [
        "alignment_tfidf_z",
        "log_review_count",
        "log_patient_count",
        "patient_need_length_main"
    ] + [c for c in X.columns if c.startswith("评论类型_")] \
      + [c for c in X.columns if c.startswith("province_group_")],

    "RF-3 Controls + Embedding":
    [
        "alignment_llm_z",
        "log_review_count",
        "log_patient_count",
        "patient_need_length_main"
    ] + [c for c in X.columns if c.startswith("评论类型_")] \
      + [c for c in X.columns if c.startswith("province_group_")],

    "RF-4 Controls + TF-IDF + Embedding":
    X.columns.tolist()
}

results = []

for name, vars_used in feature_sets.items():

    X_sub = X[vars_used]

    X_train, X_test, y_train, y_test = train_test_split(
        X_sub,
        y,
        test_size=0.30,
        random_state=42,
        stratify=y
    )

    rf = RandomForestClassifier(
        n_estimators=500,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    prob = rf.predict_proba(X_test)[:,1]

    auc = roc_auc_score(y_test, prob)

    results.append({
        "Model": name,
        "Variables": len(vars_used),
        "ROC-AUC": round(auc,3)
    })

comparison = pd.DataFrame(results)

display(comparison)

comparison.to_excel(
    "/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit/outputs/tables/Supplementary_Table_S2_Incremental_Predictive_Value.xlsx",
    index=False
)

print("Supplementary Table S2 saved.")

In [ ]:
import os
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

output_figures = "/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit/outputs/figures"
os.makedirs(output_figures, exist_ok=True)

def add_box(ax, x, y, w, h, text, fontsize=11):
    box = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.02,rounding_size=0.03",
        linewidth=1.6,
        edgecolor="black",
        facecolor="white"
    )
    ax.add_patch(box)
    ax.text(
        x + w/2, y + h/2, text,
        ha="center", va="center",
        fontsize=fontsize,
        wrap=True
    )

def add_arrow(ax, start, end):
    arrow = FancyArrowPatch(
        start, end,
        arrowstyle="->",
        mutation_scale=18,
        linewidth=1.5,
        color="black"
    )
    ax.add_patch(arrow)

fig, ax = plt.subplots(figsize=(12, 7))
ax.set_xlim(0, 12)
ax.set_ylim(0, 7)
ax.axis("off")

# Title
ax.text(
    0.3, 6.65,
    "Figure 1. Conceptual framework of semantic patient–physician alignment and patient satisfaction",
    fontsize=15,
    fontweight="bold",
    ha="left"
)

# Boxes
add_box(
    ax, 0.9, 4.75, 2.7, 1.1,
    "Patient healthcare needs\nSymptoms + review narratives"
)

add_box(
    ax, 0.9, 2.85, 2.7, 1.1,
    "Physician expertise\nSpecialty + expertise profile"
)

add_box(
    ax, 4.4, 3.65, 3.2, 1.45,
    "Semantic embedding model\nSemantic representation of\npatient needs and physician expertise"
)

add_box(
    ax, 8.5, 3.65, 2.7, 1.45,
    "Semantic alignment score\nCosine similarity"
)

add_box(
    ax, 8.5, 1.35, 2.7, 1.25,
    "Patient satisfaction\nLower-rating indicator\n(<5 stars)"
)

add_box(
    ax, 4.4, 1.0, 3.2, 1.35,
    "Control variables\nReview count, patient volume,\nconsultation type, province,\npatient text length",
    fontsize=10.5
)

# Arrows
add_arrow(ax, (3.6, 5.3), (4.4, 4.55))
add_arrow(ax, (3.6, 3.4), (4.4, 4.2))
add_arrow(ax, (7.6, 4.38), (8.5, 4.38))
add_arrow(ax, (9.85, 3.65), (9.85, 2.6))
add_arrow(ax, (7.6, 1.68), (8.5, 1.95))

# Note
ax.text(
    0.9, 0.35,
    "Note. The framework links patient-expressed healthcare needs and physician expertise through semantic alignment, "
    "while accounting for platform and contextual control variables.",
    fontsize=10,
    ha="left"
)

plt.tight_layout()

png_path = os.path.join(output_figures, "Figure1_Conceptual_Framework.png")
pdf_path = os.path.join(output_figures, "Figure1_Conceptual_Framework.pdf")

plt.savefig(png_path, dpi=300, bbox_inches="tight")
plt.savefig(pdf_path, bbox_inches="tight")

plt.show()

print("Saved:")
print(png_path)
print(pdf_path)

In [ ]:
import os
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

output_figures = "/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit/outputs/figures"
os.makedirs(output_figures, exist_ok=True)

def add_box(ax, x, y, w, h, text, fontsize=11):
    box = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.03,rounding_size=0.04",
        linewidth=1.6,
        edgecolor="black",
        facecolor="white"
    )
    ax.add_patch(box)
    ax.text(
        x + w/2, y + h/2, text,
        ha="center", va="center",
        fontsize=fontsize,
        wrap=True
    )

def add_arrow(ax, start, end):
    arrow = FancyArrowPatch(
        start, end,
        arrowstyle="->",
        mutation_scale=18,
        linewidth=1.5,
        color="black"
    )
    ax.add_patch(arrow)

fig, ax = plt.subplots(figsize=(12, 7))
ax.set_xlim(0, 12)
ax.set_ylim(0, 7)
ax.axis("off")

# Title
ax.text(
    0.6, 6.55,
    "Figure 1. Conceptual framework of semantic patient–physician alignment",
    fontsize=15,
    fontweight="bold",
    ha="left"
)

# Input boxes
add_box(
    ax, 0.8, 4.65, 2.7, 1.05,
    "Patient needs\nSymptoms + review narratives",
    fontsize=11
)

add_box(
    ax, 0.8, 2.95, 2.7, 1.05,
    "Physician profile\nSpecialty + expertise description",
    fontsize=11
)

# Embedding box
add_box(
    ax, 4.2, 3.65, 2.8, 1.25,
    "Semantic embedding model\nPatient and physician\ntext embeddings",
    fontsize=11
)

# Alignment box
add_box(
    ax, 7.8, 3.65, 2.5, 1.25,
    "Semantic alignment score\nCosine similarity",
    fontsize=11
)

# Fit box
add_box(
    ax, 7.8, 2.25, 2.5, 0.95,
    "Patient–physician fit",
    fontsize=11
)

# Outcome box
add_box(
    ax, 7.8, 0.9, 2.5, 1.0,
    "Patient satisfaction\nBinary outcome\n(rating < 5)",
    fontsize=11
)

# Controls box
add_box(
    ax, 4.2, 0.95, 2.8, 1.45,
    "Control variables\n• Review count\n• Patient volume\n• Consultation type\n• Province\n• Patient text length",
    fontsize=10
)

# Arrows
add_arrow(ax, (3.5, 5.18), (4.2, 4.35))
add_arrow(ax, (3.5, 3.48), (4.2, 4.05))
add_arrow(ax, (7.0, 4.28), (7.8, 4.28))
add_arrow(ax, (9.05, 3.65), (9.05, 3.20))
add_arrow(ax, (9.05, 2.25), (9.05, 1.90))
add_arrow(ax, (7.0, 1.65), (7.8, 1.40))

# Note
ax.text(
    0.8, 0.35,
    "Note. The framework links patient-expressed needs and physician expertise through semantic alignment, "
    "conceptualized as patient–physician fit, while accounting for platform and contextual control variables.",
    fontsize=10,
    ha="left"
)

plt.tight_layout()

png_path = os.path.join(output_figures, "Figure1_Conceptual_Framework_Final.png")
pdf_path = os.path.join(output_figures, "Figure1_Conceptual_Framework_Final.pdf")

plt.savefig(png_path, dpi=300, bbox_inches="tight")
plt.savefig(pdf_path, bbox_inches="tight")

plt.show()

print("Saved:")
print(png_path)
print(pdf_path)

In [ ]:
import os
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

output_figures = "/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit/outputs/figures"
os.makedirs(output_figures, exist_ok=True)

def add_box(ax, x, y, w, h, text, fontsize=11):
    box = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.03,rounding_size=0.04",
        linewidth=1.6,
        edgecolor="black",
        facecolor="white"
    )
    ax.add_patch(box)
    ax.text(
        x + w/2, y + h/2, text,
        ha="center", va="center",
        fontsize=fontsize,
        wrap=True
    )

def add_arrow(ax, start, end):
    arrow = FancyArrowPatch(
        start, end,
        arrowstyle="->",
        mutation_scale=18,
        linewidth=1.5,
        color="black"
    )
    ax.add_patch(arrow)

fig, ax = plt.subplots(figsize=(10, 9))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis("off")

# Title
ax.text(
    0.5, 9.55,
    "Figure 2. Study workflow",
    fontsize=15,
    fontweight="bold",
    ha="left"
)

# Workflow boxes
x = 3.0
w = 4.0
h = 0.75

steps = [
    ("Original WeDoctor dataset\n725,920 patient reviews", 8.5),
    ("Stratified analytical sample\n50,000 reviews", 7.3),
    ("Final analysis sample\n46,674 reviews with complete text fields", 6.1),
    ("Text construction and preprocessing\nPatient needs + physician expertise", 4.9),
    ("Semantic alignment measurement\nEmbedding cosine similarity + TF-IDF baseline", 3.7),
    ("Statistical modeling\nLogistic regression models", 2.5),
    ("Robustness analysis\nRandom Forest classifier", 1.3)
]

for text, y in steps:
    add_box(ax, x, y, w, h, text, fontsize=10.5)

# Arrows between boxes
for i in range(len(steps)-1):
    y_start = steps[i][1]
    y_end = steps[i+1][1]
    add_arrow(ax, (x + w/2, y_start), (x + w/2, y_end + h))

# Note
ax.text(
    0.5, 0.35,
    "Note. The workflow summarizes dataset construction, text-based semantic alignment measurement, "
    "and the main statistical and robustness analyses.",
    fontsize=10,
    ha="left"
)

plt.tight_layout()

png_path = os.path.join(output_figures, "Figure2_Study_Workflow.png")
pdf_path = os.path.join(output_figures, "Figure2_Study_Workflow.pdf")

plt.savefig(png_path, dpi=300, bbox_inches="tight")
plt.savefig(pdf_path, bbox_inches="tight")

plt.show()

print("Saved:")
print(png_path)
print(pdf_path)

In [ ]:
import os
import matplotlib.pyplot as plt

output_figures = "/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit/outputs/figures"
os.makedirs(output_figures, exist_ok=True)

# Create readable group label
df["rating_group"] = df["lower_rating"].map({
    0: "Five-star rating",
    1: "Below-5-star rating"
})

# -----------------------------
# Figure 2A: Embedding alignment
# -----------------------------

plt.figure(figsize=(7, 5))

for group in ["Five-star rating", "Below-5-star rating"]:
    subset = df.loc[df["rating_group"] == group, "alignment_llm"].dropna()
    subset.plot(kind="density", linewidth=2, label=group)

plt.xlabel("Embedding-based semantic alignment")
plt.ylabel("Density")
plt.title("Figure 2A. Distribution of embedding-based semantic alignment")
plt.legend()
plt.tight_layout()

fig2a_png = os.path.join(output_figures, "Figure2A_Embedding_Alignment_Distribution.png")
fig2a_pdf = os.path.join(output_figures, "Figure2A_Embedding_Alignment_Distribution.pdf")

plt.savefig(fig2a_png, dpi=300, bbox_inches="tight")
plt.savefig(fig2a_pdf, bbox_inches="tight")
plt.show()

# -----------------------------
# Figure 2B: TF-IDF alignment
# -----------------------------

plt.figure(figsize=(7, 5))

for group in ["Five-star rating", "Below-5-star rating"]:
    subset = df.loc[df["rating_group"] == group, "alignment_tfidf"].dropna()
    subset.plot(kind="density", linewidth=2, label=group)

plt.xlabel("TF-IDF lexical alignment")
plt.ylabel("Density")
plt.title("Figure 2B. Distribution of TF-IDF lexical alignment")
plt.legend()
plt.tight_layout()

fig2b_png = os.path.join(output_figures, "Figure2B_TFIDF_Alignment_Distribution.png")
fig2b_pdf = os.path.join(output_figures, "Figure2B_TFIDF_Alignment_Distribution.pdf")

plt.savefig(fig2b_png, dpi=300, bbox_inches="tight")
plt.savefig(fig2b_pdf, bbox_inches="tight")
plt.show()

print("Figure 2A saved:")
print(fig2a_png)
print(fig2a_pdf)

print("\nFigure 2B saved:")
print(fig2b_png)
print(fig2b_pdf)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

# Folder
output_figures = "/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit/outputs/figures"
os.makedirs(output_figures, exist_ok=True)

# ----------------------------------------
# Create prediction dataset
# ----------------------------------------

pred = df.copy()

alignment_range = np.linspace(
    pred["alignment_llm_z"].min(),
    pred["alignment_llm_z"].max(),
    100
)

predicted_prob = []

for value in alignment_range:

    tmp = pred.copy()

    tmp["alignment_llm_z"] = value

    p = model4.predict(tmp)

    predicted_prob.append(p.mean())

# ----------------------------------------
# Plot
# ----------------------------------------

plt.figure(figsize=(7,5))

plt.plot(
    alignment_range,
    predicted_prob,
    linewidth=3
)

plt.xlabel("Embedding-based semantic alignment (standardized)")
plt.ylabel("Predicted probability of below-5-star rating")
plt.title("Figure 3. Predicted probability across semantic alignment")

plt.grid(alpha=0.3)

plt.tight_layout()

png = os.path.join(
    output_figures,
    "Figure3_Predicted_Probability.png"
)

pdf = os.path.join(
    output_figures,
    "Figure3_Predicted_Probability.pdf"
)

plt.savefig(png,dpi=300,bbox_inches="tight")
plt.savefig(pdf,bbox_inches="tight")

plt.show()

print("Saved:")
print(png)
print(pdf)

In [ ]:
%who

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

output_figures = "/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit/outputs/figures"
os.makedirs(output_figures, exist_ok=True)

prediction_data = df.copy()

alignment_range = np.linspace(
    prediction_data["alignment_llm_z"].quantile(0.01),
    prediction_data["alignment_llm_z"].quantile(0.99),
    100
)

predicted_prob = []

for value in alignment_range:
    tmp = prediction_data.copy()
    tmp["alignment_llm_z"] = value
    p = m4.predict(tmp)
    predicted_prob.append(p.mean())

plt.figure(figsize=(7, 5))

plt.plot(
    alignment_range,
    predicted_prob,
    linewidth=3
)

plt.xlabel("Embedding-based semantic alignment (standardized)")
plt.ylabel("Predicted probability of below-5-star rating")
plt.title("Figure 3. Predicted probability across semantic alignment")
plt.grid(alpha=0.3)
plt.tight_layout()

png = os.path.join(output_figures, "Figure3_Predicted_Probability.png")
pdf = os.path.join(output_figures, "Figure3_Predicted_Probability.pdf")

plt.savefig(png, dpi=300, bbox_inches="tight")
plt.savefig(pdf, bbox_inches="tight")

plt.show()

print("Saved:")
print(png)
print(pdf)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

output_figures = "/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit/outputs/figures"
os.makedirs(output_figures, exist_ok=True)

# Range from 1st to 99th percentile to avoid extreme tails
alignment_range = np.linspace(
    df["alignment_llm_z"].quantile(0.01),
    df["alignment_llm_z"].quantile(0.99),
    100
)

mean_probs = []
lower_ci = []
upper_ci = []

# Simulation-based confidence interval using model parameter covariance
params = m4.params.values
cov = m4.cov_params().values

np.random.seed(42)
sim_params = np.random.multivariate_normal(params, cov, size=1000)

for value in alignment_range:
    tmp = df.copy()
    tmp["alignment_llm_z"] = value

    # Model matrix
    X_pred = m4.model.exog.copy()
    design_info = m4.model.data.design_info
    X_new = pd.DataFrame(
        m4.model.data.design_info.builder.transform(tmp),
        columns=m4.params.index
    )

    probs = []

    for b in sim_params:
        linear_pred = np.dot(X_new, b)
        p = 1 / (1 + np.exp(-linear_pred))
        probs.append(p.mean())

    probs = np.array(probs)

    mean_probs.append(probs.mean())
    lower_ci.append(np.percentile(probs, 2.5))
    upper_ci.append(np.percentile(probs, 97.5))

plt.figure(figsize=(7, 5))

plt.plot(
    alignment_range,
    mean_probs,
    linewidth=2.5,
    label="Predicted probability"
)

plt.fill_between(
    alignment_range,
    lower_ci,
    upper_ci,
    alpha=0.20,
    label="95% CI"
)

plt.xlabel("Semantic alignment score (standardized)")
plt.ylabel("Predicted probability of patient dissatisfaction")
plt.title("Figure 3. Predicted probability of below-5-star rating across semantic alignment")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()

png = os.path.join(output_figures, "Figure3_Predicted_Probability_With_CI.png")
pdf = os.path.join(output_figures, "Figure3_Predicted_Probability_With_CI.pdf")

plt.savefig(png, dpi=300, bbox_inches="tight")
plt.savefig(pdf, bbox_inches="tight")

plt.show()

print("Saved:")
print(png)
print(pdf)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from patsy import build_design_matrices

output_figures = "/Users/isaacasante/Documents/offload/Research Projects/Paper4_Patient_Physician_Fit/outputs/figures"
os.makedirs(output_figures, exist_ok=True)

alignment_range = np.linspace(
    df["alignment_llm_z"].quantile(0.01),
    df["alignment_llm_z"].quantile(0.99),
    100
)

mean_probs = []
lower_ci = []
upper_ci = []

params = m4.params.values
cov = m4.cov_params().values

np.random.seed(42)
sim_params = np.random.multivariate_normal(params, cov, size=1000)

design_info = m4.model.data.design_info

for value in alignment_range:
    tmp = df.copy()
    tmp["alignment_llm_z"] = value

    X_new = build_design_matrices(
        [design_info],
        tmp,
        return_type="dataframe"
    )[0]

    probs = []

    for b in sim_params:
        linear_pred = np.dot(X_new, b)
        p = 1 / (1 + np.exp(-linear_pred))
        probs.append(p.mean())

    probs = np.array(probs)

    mean_probs.append(probs.mean())
    lower_ci.append(np.percentile(probs, 2.5))
    upper_ci.append(np.percentile(probs, 97.5))

plt.figure(figsize=(7, 5))

plt.plot(
    alignment_range,
    mean_probs,
    linewidth=2.5,
    label="Predicted probability"
)

plt.fill_between(
    alignment_range,
    lower_ci,
    upper_ci,
    alpha=0.20,
    label="95% CI"
)

plt.xlabel("Semantic alignment score (standardized)")
plt.ylabel("Predicted probability of patient dissatisfaction")
plt.title("Figure 3. Predicted probability of below-5-star rating across semantic alignment")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()

png = os.path.join(output_figures, "Figure3_Predicted_Probability_With_CI.png")
pdf = os.path.join(output_figures, "Figure3_Predicted_Probability_With_CI.pdf")

plt.savefig(png, dpi=300, bbox_inches="tight")
plt.savefig(pdf, bbox_inches="tight")

plt.show()

print("Saved:")
print(png)
print(pdf)

In [1]:
from pathlib import Path
import pandas as pd

DATA_FILE = Path("../data/paper4_final_regression_dataset.csv")

if not DATA_FILE.exists():
    raise FileNotFoundError(
        "The analysis dataset is not included in this repository. "
        "Place an appropriately obtained dataset at "
        "'data/paper4_final_regression_dataset.csv'."
    )

df = pd.read_csv(DATA_FILE)

FileNotFoundError: The analysis dataset is not included in this repository. Place an appropriately obtained dataset at 'data/paper4_final_regression_dataset.csv'.

In [2]:
from pathlib import Path
import pandas as pd

DATA_FILE = Path("../data/paper4_final_regression_dataset.csv")

if DATA_FILE.exists():
    df = pd.read_csv(DATA_FILE)

    print("Dataset loaded successfully.")
    print("Rows:", df.shape[0])
    print("Columns:", df.shape[1])

else:
    df = None

    print(
        "Dataset not found.\n"
        "The source dataset is not included in this public repository.\n"
        "To run the analysis, place an appropriately obtained file at:\n"
        "../data/paper4_final_regression_dataset.csv"
    )

Dataset not found.
The source dataset is not included in this public repository.
To run the analysis, place an appropriately obtained file at:
../data/paper4_final_regression_dataset.csv


### Data Access Note

The source dataset used in this study is not included in this public repository because redistribution may be restricted by the WeDoctor platform's terms of use. This notebook documents the complete analytical workflow. Researchers with appropriately obtained data may reproduce the analyses by placing the processed dataset in the expected data directory.